**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/fritsche-2022"], check=True)
    os.chdir("/content/encore/entries/fritsche-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Fritsche, Solomon & de Lange (2022) on DANDI:000021

Paper: Fritsche, Solomon & de Lange (2022), "Brief Stimuli Cast a Persistent Long-Term Trace in
Visual Cortex", J Neurosci 42(10):1999-2010, doi:10.1523/JNEUROSCI.1350-21.2021. The paper reuses
the Allen Brain Observatory Visual Coding Neuropixels data.

Dataset: DANDI:000021 (Allen Institute, Visual Coding, Neuropixels, Brain Observatory 1.1
stimulus set), version 0.251116.2246, doi:10.48324/dandi.000021/0.251116.2246. The paper names
the dandiset but no version. This is the only published version, and it is the most recent;
the draft has the same 214 assets and size. It holds the 32 sessions (one per mouse) the paper
describes, so no earlier version could match the claims file better.

Claims: claims/fritsche-2022.yaml, written from the version of record (PMC8916757; no
supplement). The paper's code (doi:10.34973/kpt9-y956) is restricted and was not read.

Data are streamed from the archive; nothing is downloaded. Loading reduces each session to spike
counts per unit and presentation (about 2 to 3 minutes for the 32 files with four processes).

| Claim | Tested |
|---|---|
| units-responsive-count | tested |
| fig1d-v1-1back | tested |
| fig1d-extrastriate-1back | tested |
| fig1d-thalamus-1back | tested |
| fig1e-consistent-across-mice | tested |
| fig2-v1-tuning-main-effect | tested |
| fig2-v1-tuning-interaction | tested |
| fig2-v1-adaptation-at-all-orientations | tested |
| fig2-thalamus-no-tuning-dependence | tested |
| fig3a-v1-long-term | tested |
| fig3a-v1-double-exponential | tested |
| fig3a-v1-half-life | tested |
| fig4-cumulative-adaptation | tested |
| fig4b-linear-accumulation | tested |
| fig3b-thalamus-short-term | tested |
| v1-vs-thalamus-welch | tested |
| fig3c-mice-long-term | tested |
| decay-cortex-double-better | tested |
| table1-extrastriate-tau-slow | tested |
| decay-thalamus-single | tested |
| fig5-repeat-suppression | tested |
| fig6c-static-v1-1back | tested |
| fig6c-static-other-areas-1back | tested |
| fig6d-static-consistent-across-mice | tested |
| fig7a-static-v1-long-term | tested |
| fig7b-static-cortex-decay | tested |
| fig7-static-lgn-single | tested |
| fig7b-static-areas-long-term | tested |
| fig7c-static-long-term-across-mice | tested |
| fig8-no-spurious-long-term | tested |
| robust-threshold-preferred-window | tested |

No claim is blocked.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import itertools, resource, time, zlib, warnings
import numpy as np
import pandas as pd
from scipy import stats
import adaptation as A

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
KEY = "fritsche-2022"
def seed(claim):
    return zlib.crc32(f"{KEY}/{claim}".encode())
T0 = time.time()

In [ ]:
assets = A.session_assets()
print(len(assets), "session files at version", A.VERSION)
S = A.load_all(assets)
print(f"loaded in {time.time() - T0:.0f} s; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

The dandiset's 214 assets are 32 session files and 182 per-probe LFP files; only the session
files are used. Each session file holds the units table (with the AllenSDK quality metrics),
the electrodes table (with the CCF area of each channel), and the stimulus tables of every
stimulus, of which the drifting and static grating tables and the spontaneous (gray screen)
epochs are used.

In [ ]:
rows = []
for s in S:
    m, dg, sg, u = s["meta"], s["dg"], s["sg"], s["units"]
    rows.append(dict(
        subject=m["subject_id"], session=m["session_id"], genotype=m["genotype"].split("/")[0],
        sex=m["sex"], age=m["age"], units=len(u), good=int((u.quality == "good").sum()),
        pass_metrics=int(u.passes_quality.sum()),
        dg_rows=len(dg), dg_blank=int(dg.orientation.isna().sum()),
        dg_blocks=dg.stimulus_block.nunique(), sg_rows=len(sg),
        sg_blank=int(sg.orientation.isna().sum()), sg_blocks=sg.stimulus_block.nunique(),
        spont_s=round(float((s["spont"].stop_time - s["spont"].start_time).sum())),
        invalid_intervals=len(s["invalid"])))
sess = pd.DataFrame(rows)
print(sess.to_string())
print("\nsubjects", sess.subject.nunique(), "sessions", sess.session.nunique())
print(sess.genotype.value_counts().to_string())
print("\nsex", sess.sex.value_counts().to_dict())

In [ ]:
# Stimulus parameters, durations and spacing.
dg_all = pd.concat([s["dg"].assign(sess=i) for i, s in enumerate(S)])
sg_all = pd.concat([s["sg"].assign(sess=i) for i, s in enumerate(S)])
print("drifting directions", sorted(dg_all.orientation.dropna().unique()))
print("temporal frequencies", sorted(dg_all.temporal_frequency.dropna().unique()))
print("contrast", sorted(dg_all.contrast.dropna().unique()))
dur = dg_all.stop_time - dg_all.start_time
print(f"drifting duration {dur.min():.3f} to {dur.max():.3f} s")
isi = np.concatenate([np.diff(s["dg"].sort_values("start_time").start_time) for s in S])
print(f"drifting onset spacing within blocks: median {np.median(isi):.3f} s, "
      f"10th to 90th percentile {np.percentile(isi, 10):.3f} to {np.percentile(isi, 90):.3f} s")
print("static orientations", sorted(sg_all.orientation.dropna().unique()))
print("static spatial frequencies", sorted(sg_all.spatial_frequency.dropna().unique()))
print("static phases", sorted(sg_all.phase.dropna().unique()))
sd = sg_all.stop_time - sg_all.start_time
print(f"static duration {sd.min():.3f} to {sd.max():.3f} s")
gaps = np.concatenate([np.diff(s["sg"].sort_values("start_time").start_time.values) for s in S])
print(f"static onset spacing: median {np.median(gaps):.4f} s, 99th percentile "
      f"{np.percentile(gaps, 99):.4f} s, {np.sum(gaps > 1)} gaps over 1 s (between blocks)")
per = dg_all.dropna(subset=["orientation"]).groupby(["sess", "orientation", "temporal_frequency"]).size()
print("drifting presentations per direction and frequency per session:", per.min(), "to", per.max())
cnt = sg_all.dropna(subset=["orientation"]).groupby(["sess", "orientation", "spatial_frequency", "phase"]).size()
print("static presentations per condition per session:", cnt.min(), "to", cnt.max())
for i in [0, 1, 2]:
    d = S[i]["dg"].dropna(subset=["orientation"])
    print(sess.session[i], "drifting gratings", len(d), d.groupby("orientation").size().to_dict())

In [ ]:
# Invalid intervals and whether they overlap the grating epochs.
for i, s in enumerate(S):
    inv = s["invalid"]
    if len(inv) == 0:
        continue
    ep = {"dg": (s["dg"].start_time.min(), s["dg"].stop_time.max()),
          "sg": (s["sg"].start_time.min(), s["sg"].stop_time.max())}
    ov = {k: int(((inv.start_time < b) & (inv.stop_time > a)).sum()) for k, (a, b) in ep.items()}
    print(sess.session[i], len(inv), "invalid intervals, overlapping drifting", ov["dg"],
          "static", ov["sg"], "total length", round(float((inv.stop_time - inv.start_time).sum())), "s")

In [ ]:
# Unit areas and the AllenSDK metrics.
U = pd.concat([s["units"].assign(sess=i) for i, s in enumerate(S)])
print("units", len(U), "quality labels", U.quality.value_counts().to_dict())
for c in ["isi_violations", "amplitude_cutoff", "presence_ratio", "firing_rate"]:
    v = U[c].replace([np.inf, -np.inf], np.nan)
    print(f"{c}: {v.min():.3g} to {v.max():.3g}, NaN {int(v.isna().sum())}")
print("units with no area label in the electrodes table", int(U.location.isna().sum()))
q = U[U.passes_quality & (U.quality == "good")]
print("pass metrics and labelled good:", len(q))
print(q.location.value_counts().head(30).to_string())

In [ ]:
# Recordings deposited more than once: identical stimulus timing, or identical spike count
# vectors of units across or within sessions.
t0 = [round(float(s["dg"].start_time.min()), 4) for s in S]
print("distinct first drifting onsets", len(set(t0)), "of", len(S))
h = {}
dup_within, dup_across = 0, 0
for i, s in enumerate(S):
    seen = set()
    for k in range(s["dg_counts"].shape[0]):
        v = np.r_[s["dg_counts"][k], s["sg_counts"][k]]
        if v.sum() < 50:
            continue
        key = hash(v.tobytes())
        if key in seen:
            dup_within += 1
        seen.add(key)
        if key in h and h[key] != i:
            dup_across += 1
        h.setdefault(key, i)
print("units with identical count vectors: within a session", dup_within, "across sessions", dup_across)

In [ ]:
# Clock check. In each session, the orientation tuning of V1 units is estimated from even trials
# and tested on odd trials, with the true orientation labels and with labels shifted by whole
# presentations. A response locked to the table's onsets gives a positive index with the true
# labels only. The onset response (0 to 100 ms against the 100 ms before onset) is also given.
def tuning_index(c, ori, shift):
    o = np.roll(ori, shift)
    ok = ~np.isnan(o)
    idx = np.flatnonzero(ok)
    ev, od = idx[::2], idx[1::2]
    oris = np.unique(o[ok])
    me = np.stack([c[:, ev][:, o[ev] == x].mean(1) for x in oris], 1)
    mo = np.stack([c[:, od][:, o[od] == x].mean(1) for x in oris], 1)
    k = me.argmax(1)
    pref = mo[np.arange(len(k)), k]
    orth = mo[np.arange(len(k)), (k + len(oris) // 2) % len(oris)]
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.nanmean((pref - orth) / (pref + orth))
clock = []
for i, s in enumerate(S):
    v1 = (s["units"].location.values[s["sel"]] == "VISp")
    if v1.sum() < 5:
        clock.append(dict(session=sess.session[i], v1_units=int(v1.sum())))
        continue
    dg = s["dg"]
    order = np.argsort(dg.start_time.values)
    ori = np.mod(dg.orientation.values[order], 180)
    c = s["dg_counts"][v1][:, order].astype(float)
    sgo = np.argsort(s["sg"].start_time.values)
    sori = np.mod(s["sg"].orientation.values[sgo], 180)
    cs = s["sg_counts"][v1][:, sgo].astype(float)
    on = s["dg_counts"][v1][:, ~np.isnan(dg.orientation.values)].mean()
    pre = s["dg_pre"][v1][:, ~np.isnan(dg.orientation.values)].mean()
    clock.append(dict(session=sess.session[i], v1_units=int(v1.sum()),
                      dg_true=tuning_index(c, ori, 0), dg_shift_p1=tuning_index(c, ori, 1),
                      dg_shift_m1=tuning_index(c, ori, -1),
                      sg_true=tuning_index(cs, sori, 0), sg_shift_p10=tuning_index(cs, sori, 10),
                      sg_shift_m10=tuning_index(cs, sori, -10),
                      onset_spikes=on, pre_spikes=pre))
clock = pd.DataFrame(clock)
print(clock.round(3).to_string())
c_ = clock.dropna()
m_dg = c_.dg_true - c_[["dg_shift_p1", "dg_shift_m1"]].max(axis=1)
m_sg = c_.sg_true - c_[["sg_shift_p10", "sg_shift_m10"]].max(axis=1)
print(f"sessions checked {len(c_)}; drifting: true labels above both shifts in {int((m_dg > 0).sum())}, "
      f"margin {m_dg.min():.3f} to {m_dg.max():.3f}; static: in {int((m_sg > 0).sum())}, margin "
      f"{m_sg.min():.3f} to {m_sg.max():.3f}; onset spikes above pre-onset in {int((c_.onset_spikes > c_.pre_spikes).sum())}")

Checks of the description against the data. The files hold 32 mice, one session each: 16 wild type, 6 Sst-IRES-Cre, 5 Pvalb-IRES-Cre and 5 Vip-IRES-Cre, as the paper says. Drifting gratings have the eight directions, five temporal frequencies and contrast 0.8 of the Methods, last 1.985 to 2.018 s and start every 3.003 s, in three blocks with 30 blank trials per session. Three sessions (715093703, 719161530, 721123822) hold 598 gratings, not 600; one or two conditions there have 14 presentations. Static gratings have the six orientations, five spatial frequencies and four phases, last 0.200 to 0.317 s (median spacing 0.2502 s) with no gap within a block, and there are 189 to 197 blank sweeps among 6000 presentations, about one in 31 (the Methods say roughly one in 25). The units table holds every sorted unit, 74658 in all, of which 20182 are labelled `noise` in a `quality` column that has no description; the AllenSDK quality metrics are in columns without descriptions or units. Every unit's peak channel has an area label; 3723 of the units that pass the metrics are on channels labelled `grey` and 686 on `VIS`, which name no area. No recording is deposited twice: the first drifting onsets differ in all 32 sessions, and no unit's count vector repeats within or across sessions. Thirteen sessions list invalid intervals, tagged with a probe, some of which overlap the grating epochs; the paper does not mention them and they are not applied here. A variant that leaves out the presentations inside them is shown after fig3b-thalamus-short-term. The clock of the spike times and the stimulus tables is consistent with one clock: in every session the V1 orientation tuning measured with the true labels exceeds that with labels shifted by one drifting or ten static presentations, and the 0 to 100 ms count exceeds the count in the 100 ms before onset.

## Unit selection

Units pass the AllenSDK quality metrics (ISI violations <= 0.5, amplitude cutoff <= 0.1,
presence ratio >= 0.9; Methods, "Unit exclusion"). The NWB files also label 20182 of the 74658
units `noise`; the AllenSDK's units table leaves these out by default, which the Methods'
"default quality metrics of the AllenSDK" is read to include (an assumption; the variant
without it is shown). Responsiveness follows Methods, "Unit exclusion", with the readings of O4.

In [ ]:
def select(s, stim, rule="mean_plus_sd", src="blank", by="orientation", thr=5.0, good=True,
           win=None):
    if stim == "dg":
        c, sp, w = s["dg_counts"], s["sp100"], A.DG_WIN
    elif win == A.SG_WIN_SHORT:
        c, sp, w = s["sg_counts100"], s["sp100"], A.SG_WIN_SHORT
    else:
        c, sp, w = s["sg_counts"], s["sp250"], A.SG_WIN
    table = s[stim]
    pref_ori, pr, by_ori, oris = A.preferred(c, table, w, by)
    gm, gs = A.gray_stats(c, table, sp, w, src)
    r = A.responsive(pr, gm, gs, thr, rule)
    if good:
        r &= s["units"].quality.values[s["sel"]] == "good"
    return r, pref_ori, by_ori, oris

def count_regions(stim, **kw):
    loc = []
    for s in S:
        r = select(s, stim, **kw)[0]
        loc.append(s["units"].location.values[s["sel"]][r])
    return pd.Series(np.concatenate(loc)).value_counts()

## units-responsive-count

> The drifting grating analysis rests on 2365 visually responsive neurons in visual cortex and
> thalamus of 32 mice, with 562 in V1, 140 in LGN and 90 in LP.

Every reading of O4 is run: (a) over gray mean + 1 SD or (b) over 1 SD; gray from (c) the
blank trials or (d) the spontaneous epochs (100 ms bins); preferred orientation (e) pooled over
directions and temporal frequencies or (f) the best single condition. Each also without the
`noise` label filter. Unlabelled channels (`grey`, `VIS`) are not regions the paper names and are
listed apart. "Kept" is the set of the paper's eight regions with at least 50 units.

In [ ]:
NAMED = set(A.AREA_NAMES)
rows = []
for good in [True, False]:
    for rule, src, by in itertools.product(["mean_plus_sd", "sd"], ["blank", "spont"],
                                           ["orientation", "condition"]):
        vc = count_regions("dg", rule=rule, src=src, by=by, good=good)
        named = vc[vc.index.isin(NAMED)]
        kept = named[named >= 50]
        other50 = sorted(vc[(~vc.index.isin(NAMED)) & (vc >= 50)].index)
        g = lambda k: int(named.get(k, 0))
        tot = int(kept.sum())
        ok = (abs(tot - 2365) <= 236.5 and abs(g("VISp") - 562) <= 56.2
              and abs(g("LGd") - 140) <= 14 and abs(g("LP") - 90) <= 9 and len(kept) == 8)
        rows.append(dict(good_only=good, O4=f"{'a' if rule == 'mean_plus_sd' else 'b'}"
                         f"{'c' if src == 'blank' else 'd'}{'e' if by == 'orientation' else 'f'}",
                         total=tot, V1=g("VISp"), LM=g("VISl"), AL=g("VISal"), RL=g("VISrl"),
                         PM=g("VISpm"), AM=g("VISam"), LGN=g("LGd"), LP=g("LP"),
                         named_kept=len(kept), other_labels_50=",".join(other50),
                         all_within_tolerance=ok))
counts_tab = pd.DataFrame(rows)
print(counts_tab.to_string())
print("\npaper: total 2365 (tolerance 236.5), V1 562 (56.2), LGN 140 (14), LP 90 (9)")

The analyses below use the reading that the counts single out for both stimulus sets (see also
the static counts below): `noise` units left out, gray mean + 1 SD of the blank presentations,
preferred orientation pooled over the other stimulus parameters (O4 a, c, e).

Ratios use the main reading of O1 to O3: per unit the mean rate over all repeat trials over the
mean over all orthogonal trials (O1a); the region value is exp(mean log ratio) (O2a); blank
trials hold a position in the sequence and pairs are formed within each stimulus block (O3a, c).
The other readings are shown for the 1-back values.

In [ ]:
MAIN = dict(rule="mean_plus_sd", src="blank", by="orientation", good=True)
def build(stim, sel_kw=MAIN, blanks="hold", blocks="within", n_max=50, win=None, short_counts=False):
    """Per responsive unit: area, session index, n-back structures and preferred orientation."""
    out = dict(area=[], sess=[], pref=[], nb=[], by_ori=[], oris=None)
    for i, s in enumerate(S):
        r, pref, by_ori, oris = select(s, stim, win=win, **sel_kw)
        area = s["units"].area.values[s["sel"]]
        r &= pd.notna(area)
        if stim == "dg":
            c = s["dg_counts"]
        else:
            c = s["sg_counts100"] if (win == A.SG_WIN_SHORT or short_counts) else s["sg_counts"]
        nb, nb_oris = A.nback_rates(c[r], s[stim], n_max, blanks, blocks)
        out["area"].append(area[r]); out["sess"].append(np.full(r.sum(), i))
        out["pref"].append(pref[r]); out["nb"].append(nb); out["by_ori"].append(by_ori[r])
        out["oris"] = nb_oris
    out["area"] = np.concatenate(out["area"]); out["sess"] = np.concatenate(out["sess"])
    out["pref"] = np.concatenate(out["pref"]); out["by_ori"] = np.concatenate(out["by_ori"])
    out["nb"] = {k: np.concatenate([nb[k] for nb in out["nb"]]) for k in ["rs", "rn", "os", "on"]}
    return out

t = time.time()
D = {}
for blanks, blocks in itertools.product(["hold", "remove"], ["within", "across"]):
    D[("dg", blanks, blocks)] = build("dg", blanks=blanks, blocks=blocks)
    D[("sg", blanks, blocks)] = build("sg", blanks=blanks, blocks=blocks)
print(f"n-back tables built in {time.time() - t:.0f} s")
DG, SG = D[("dg", "hold", "within")], D[("sg", "hold", "within")]
AR = {k: {h: A.ratios(v["nb"], h) for h in ["pooled", "per_ori", "per_ori_geo"]} for k, v in D.items()}
ar_dg, ar_sg = AR[("dg", "hold", "within")]["pooled"], AR[("sg", "hold", "within")]["pooled"]
for name, ar in [("drifting", ar_dg), ("static", ar_sg)]:
    lr = np.log(ar)
    print(f"{name}: {ar.shape[0]} units x {ar.shape[1]} n; non-finite log ratios {int((~np.isfinite(lr)).sum())}")
    print(f"  ratio range {np.nanmin(ar):.3f} to {np.nanmax(ar):.3f}")
    print("  units per area", pd.Series(D[('dg' if name == 'drifting' else 'sg', 'hold', 'within')]['area']).value_counts().to_dict())
print("repeat pairs per unit at 1-back (drifting), median", np.median(DG["nb"]["rn"][:, 0].sum(1)),
      "orthogonal", np.median(DG["nb"]["on"][:, 0].sum(1)))
print("repeat pairs per unit at 1-back (static), median", np.median(SG["nb"]["rn"][:, 0].sum(1)),
      "orthogonal", np.median(SG["nb"]["on"][:, 0].sum(1)))

def onebak_table(stim):
    rows = []
    for (st, blanks, blocks), d in D.items():
        if st != stim:
            continue
        for how in ["pooled", "per_ori", "per_ori_geo"]:
            ar = AR[(st, blanks, blocks)][how]
            for agg in ["geo", "mean"]:
                row = dict(O1=how, O2=agg, blanks=blanks, blocks=blocks)
                for a in A.CORTEX + A.THALAMUS:
                    x = ar[d["area"] == a, 0]
                    row[a] = round(A.region_value(x, agg), 3) if len(x) else np.nan
                rows.append(row)
    return pd.DataFrame(rows)

def region_stats(ar, area, n=0, regions=A.CORTEX + A.THALAMUS):
    rows = []
    for a in regions:
        x = ar[area == a, n]
        lr = np.log(x); lr = lr[np.isfinite(lr)]
        if len(lr) < 2:
            continue
        t_, p, k = A.ttest_log(x)
        ci = np.exp(lr.mean() + np.array([-1, 1]) * stats.t.ppf(0.975, k - 1) * lr.std(ddof=1) / np.sqrt(k))
        rows.append(dict(area=a, n_units=k, ratio=np.exp(lr.mean()), ci_low=ci[0], ci_high=ci[1],
                         t=t_, p=p))
    df = pd.DataFrame(rows)
    df["p_fdr"] = A.fdr(df.p.values)[0]
    return df

In [ ]:
# Variant: the main analyses with `noise` units kept (O4 a, c, e otherwise), to show how much the
# adopted filter matters downstream.
NOISE_KW = dict(MAIN, good=False)
dgv = build("dg", sel_kw=NOISE_KW); sgv = build("sg", sel_kw=NOISE_KW)
arv, arsv = A.ratios(dgv["nb"], "pooled"), A.ratios(sgv["nb"], "pooled")
print("drifting 1-back, noise units kept:", {a: round(A.region_value(arv[dgv["area"] == a, 0]), 3) for a in A.CORTEX + A.THALAMUS})
print("static 1-back, noise units kept:", {a: round(A.region_value(arsv[sgv["area"] == a, 0]), 3) for a in A.CORTEX + A.THALAMUS})
for region in ["V1", "LGN", "LP"]:
    lr = np.log(arv[dgv["area"] == region]); lr = np.where(np.isfinite(lr), lr, np.nan)
    ps = [stats.ttest_1samp(lr[:, n][np.isfinite(lr[:, n])], 0).pvalue for n in range(10)]
    adj, rej = A.fdr(ps)
    m = np.nanmean(lr, 0)
    sig = [n + 1 for n in range(10) if rej[n] and m[n] < 0]
    dd, ss = A.fit_models(m, -1)
    print(f"{region}, noise units kept: {int((dgv['area'] == region).sum())} units; significant n {sig}; "
          f"double fit a_fast {100 * dd[0][0]:.2f}%, tau_fast {dd[0][1]:.3f}, a_slow {100 * dd[0][2]:.2f}%, "
          f"tau_slow {dd[0][3]:.2f}; single tau {ss[0][1]:.3f}")

**Result: reproduced.** With `noise` units left out and the reading O4 a, c, e (over the gray mean + 1 SD of the blank trials, preferred orientation pooled), 2422 units are responsive in the eight regions (V1 580, LGN 142, LP 91), each count within tolerance, and all eight named regions have at least 50. Without the `noise` filter, reading a, d, e also matches (2298; V1 548, LGN 153, LP 92). Units on channels labelled `grey` or `VIS` also reach 50; they name no region and are not counted as regions. With `noise` units kept, the 1-back ratios of the eight regions move by at most 0.007 for either stimulus set and V1 stays significant to 8-back with a tau_slow of 6.93, but LGN gains a significant 7-back and its single-exponential tau falls to 0.000; the claims on the thalamic decay rest on the filter. Leaving out the presentations inside the files' invalid intervals (375 drifting presentations in 5 sessions, 7001 static in 10; variant after fig3b-thalamus-short-term) gives 2425 drifting-responsive units (V1 585, LGN 140, LP 94), all within tolerance.

## fig1d-v1-1back

> In V1 the immediate repetition of orientation reduces the early (0 to 100 ms) response to
> drifting gratings by 17%, a 1-back adaptation ratio of 0.83.

In [ ]:
r1 = region_stats(ar_dg, DG["area"])
def show_p(df):
    out = df.copy()
    for c in [c for c in out.columns if c.startswith("p")]:
        out[c] = out[c].map(lambda x: f"{x:.2g}")
    return out.round(4).to_string()
print("main reading (O1a, O2a, O3a+c); CI is the t interval of the mean log ratio")
print(show_p(r1))
tab1 = onebak_table("dg")
print("\n1-back ratio under every reading of O1 to O3 (drifting)")
print(tab1.to_string())
v1 = tab1["V1"]
print(f"\nV1 range over readings {v1.min():.3f} to {v1.max():.3f}; readings inside [0.81, 0.84]: "
      f"{int(((v1 >= 0.81) & (v1 <= 0.84)).sum())} of {len(v1)}")
for (st, bl, bk), d in D.items():
    if st == "dg":
        for how in ["pooled", "per_ori", "per_ori_geo"]:
            t_, p, k = A.ttest_log(AR[(st, bl, bk)][how][d["area"] == "V1", 0])
            print(f"V1 t test, {how}, {bl}, {bk}: t = {t_:.2f}, p = {p:.1e}, n = {k}")
print("by session: V1 ratio per mouse (units)")
v1m = pd.Series(ar_dg[DG["area"] == "V1", 0]).groupby(DG["sess"][DG["area"] == "V1"]).apply(
    lambda x: f"{np.exp(np.log(x).mean()):.3f} ({len(x)})")
print(v1m.to_dict())

**Result: reproduced.** Under the main reading the V1 1-back ratio is 0.821 (t interval 0.804 to 0.839) over 580 units, t = -18.02, p = 6.0e-58, inside [0.81, 0.84]. Four of the 24 readings fall inside the interval, all with the pooled per-unit ratio (O1a) and the geometric region value (O2a); the per-orientation readings give 0.849 to 0.931.

## fig1d-extrastriate-1back

> Extrastriate areas show similar 1-back adaptation to drifting gratings, with ratios between
> 0.80 and 0.88, all significant.

Correction (O6): Benjamini-Hochberg over the eight areas (`p_fdr` above); Bonferroni over five
is shown as a variant.

In [ ]:
ex = r1[r1.area.isin(["LM", "AL", "RL", "PM", "AM"])].copy()
ex["p_bonf5"] = np.minimum(ex.p * 5, 1)
print(show_p(ex))
print(f"\nmain reading: range {ex.ratio.min():.3f} to {ex.ratio.max():.3f}; all in [0.72, 0.968]: "
      f"{bool(ex.ratio.between(0.72, 0.968).all())}; largest corrected p {ex.p_fdr.max():.1e}")
rng_tab = tab1[["O1", "O2", "blanks", "blocks"]].copy()
rng_tab["ex_min"] = tab1[["LM", "AL", "RL", "PM", "AM"]].min(axis=1)
rng_tab["ex_max"] = tab1[["LM", "AL", "RL", "PM", "AM"]].max(axis=1)
rng_tab["all_in_range"] = (rng_tab.ex_min >= 0.72) & (rng_tab.ex_max <= 0.968)
print(rng_tab.to_string())

**Result: reproduced.** The five extrastriate ratios run from 0.795 (PM) to 0.876 (RL), inside [0.72, 0.968], and the largest corrected p is 2.0e-24. Every reading keeps all five inside the range except the four that average per-orientation ratios and take the arithmetic region mean.

## fig1d-thalamus-1back

> LGN and LP also show orientation-specific 1-back adaptation to drifting gratings, with ratios
> of 0.93 and 0.83.

In [ ]:
th = r1[r1.area.isin(A.THALAMUS)]
print(show_p(th))
print("\npaper: LGN 0.93 [0.91, 0.96] p = 8e-7 (n = 140); LP 0.83 [0.79, 0.88] p = 3e-10 (n = 90)")
print(tab1[["O1", "O2", "blanks", "blocks", "LGN", "LP"]].to_string())
both = tab1.LGN.between(0.91, 0.96) & tab1.LP.between(0.79, 0.88)
print("readings with both inside their CIs:", int(both.sum()), "of", len(tab1))

**Result: reproduced.** LGN 0.935 (t interval 0.912 to 0.959, 142 units) and LP 0.833 (0.793 to 0.876, 91 units) lie in the paper's intervals, both below 1 at p of the order of 1e-7 and 1e-10 (the table above). Twenty of the 24 readings put both inside.

## fig1e-consistent-across-mice

> The 1-back response reduction is consistent across mice, in thalamus and in cortex.

Per mouse (one session each), the mean over its responsive units of the 0 to 100 ms rate on
1-back repeat and orthogonal trials (Figure 1E legend). Readings: (a) every mouse has orthogonal
above repeat; (b) a two-sided sign test across mice at p < 0.05.

In [ ]:
def mouse_rates(d, group, ns, win):
    """Per session: mean over units of the repeat and orthogonal rates, trials pooled over the
    n in `ns` (pooled) or rates averaged over n (averaged)."""
    nb = d["nb"]; idx = np.array(ns) - 1
    out = []
    for pooled in [True, False]:
        if pooled:
            rep = nb["rs"][:, idx].sum((1, 2)) / nb["rn"][:, idx].sum((1, 2)) / win
            orth = nb["os"][:, idx].sum((1, 2)) / nb["on"][:, idx].sum((1, 2)) / win
        else:
            rep = (nb["rs"][:, idx].sum(2) / nb["rn"][:, idx].sum(2)).mean(1) / win
            orth = (nb["os"][:, idx].sum(2) / nb["on"][:, idx].sum(2)).mean(1) / win
        m = np.isin(d["area"], group)
        df = pd.DataFrame(dict(sess=d["sess"][m], rep=rep[m], orth=orth[m]))
        g = df.groupby("sess").agg(rep=("rep", "mean"), orth=("orth", "mean"), units=("rep", "size"))
        g["pooled"] = pooled
        out.append(g)
    return out

def consistency(g, label):
    above = int((g.orth > g.rep).sum()); n = len(g)
    p = stats.binomtest(above, n, 0.5).pvalue if n else np.nan
    ok_a = above == n and n > 0
    ok_b = (p < 0.05) and above > n / 2
    print(f"{label}: {n} mice, {above} with orthogonal above repeat, sign test p = {p:.2g}, "
          f"median orth - rep {np.median(g.orth - g.rep):.3f} Hz; (a) {ok_a}, (b) {ok_b}")
    for k, row in g[g.orth <= g.rep].iterrows():
        print(f"  not above: session {S[k]['meta']['session_id']}, {int(row.units)} units, "
              f"repeat {row.rep:.2f} Hz, orthogonal {row.orth:.2f} Hz")
    return ok_a, ok_b

In [ ]:
for grp, name in [(A.THALAMUS, "thalamus"), (A.CORTEX, "cortex")]:
    g = mouse_rates(DG, grp, [1], A.DG_WIN)[0]
    consistency(g, f"drifting 1-back, {name}")
    print(f"  rates {g[['rep', 'orth']].min().min():.2f} to {g[['rep', 'orth']].max().max():.2f} Hz")

**Result: partially reproduced.** In cortex all 32 mice have the orthogonal rate above the repeat rate (sign test p = 4.7e-10), reading (a). In thalamus 21 of the 23 mice with thalamic units do (p = 6.6e-05), so only reading (b) holds there. The two mice that fail (a) have 4 and 2 thalamic units, with repeat rates 0.36 and 0.97 Hz above orthogonal.

## fig2-v1-tuning-main-effect

> In V1, 1-back adaptation is strongest when adaptor and test match the neuron's preferred
> orientation and weakens for less preferred orientations, a main effect of relative
> orientation.

Per region, OSI from the 0 to 100 ms rates to the preferred orientation and its orthogonal
(Methods, equation 4), split into tertiles. Per unit, the 1-back ratio of each test orientation
preceded by itself over preceded by its orthogonal, at 0, 45 and 90 degrees from preferred.
Readings: ANOVA on (a) log ratios or (b) ratios; the 45 degree level (c) pools the trials of +45
and -45 or (d) averages their two ratios.

In [ ]:
def tuning_cells(d, region):
    m = d["area"] == region
    nb = {k: v[m][:, 0, :] for k, v in d["nb"].items()}   # 1-back, per current orientation
    oris = d["oris"]; k = len(oris)
    by_ori = d["by_ori"][m]
    pi = np.array([int(np.flatnonzero(oris == p)[0]) for p in d["pref"][m]])
    rows = np.arange(m.sum())
    def ratio_at(j_list, pool=True):
        if pool:
            rs = sum(nb["rs"][rows, j] for j in j_list); rn = sum(nb["rn"][rows, j] for j in j_list)
            os_ = sum(nb["os"][rows, j] for j in j_list); on = sum(nb["on"][rows, j] for j in j_list)
            return (rs / rn) / (os_ / on)
        return np.mean([(nb["rs"][rows, j] / nb["rn"][rows, j]) / (nb["os"][rows, j] / nb["on"][rows, j])
                        for j in j_list], 0)
    half = k // 2
    out = {}
    for pool in [True, False]:
        r0 = ratio_at([pi])
        r45 = ratio_at([(pi + 1) % k, (pi - 1) % k], pool)
        r90 = ratio_at([(pi + half) % k])
        out[pool] = np.stack([r0, r45, r90], 1)
    pref = by_ori[rows, pi]; orth = by_ori[rows, (pi + half) % k]
    osi = (pref - orth) / (pref + orth)
    tert = pd.qcut(pd.Series(osi).rank(method="first"), 3, labels=["low", "medium", "high"]).values
    return out, osi, np.asarray(tert)

ANOVA = {}
CELLS = {}
for region in A.CORTEX + A.THALAMUS:
    with np.errstate(divide="ignore", invalid="ignore"):
        out, osi, tert = tuning_cells(DG, region)
    for pool in [True, False]:
        R = out[pool]
        with np.errstate(divide="ignore", invalid="ignore"):
            ok = np.all(np.isfinite(np.log(R)), 1)
        CELLS[(region, pool)] = (R[ok], tert[ok], int((~ok).sum()))
        for scale in ["log", "ratio"]:
            Y = np.log(R[ok]) if scale == "log" else R[ok]
            ANOVA[(region, pool, scale)] = A.mixed_anova(Y, tert[ok])
def anova_table(regions):
    rows = []
    for (region, pool, scale), r in ANOVA.items():
        if region not in regions:
            continue
        R, tert, dropped = CELLS[(region, pool)]
        gm = np.exp(np.log(R).mean(0))
        rows.append(dict(region=region, level45="pooled (c)" if pool else "averaged (d)", scale=scale,
                         units=len(R), dropped=dropped,
                         F_rel=r["within"][0], p_rel=f"{r['within'][1]:.2g}",
                         F_int=r["interaction"][0], p_int=f"{r['interaction'][1]:.2g}",
                         df=f"{r['within'][2]},{r['within'][3]}",
                         ratio0=gm[0], ratio45=gm[1], ratio90=gm[2]))
    return pd.DataFrame(rows)

In [ ]:
print(anova_table(["V1"]).round(3).to_string())
R, tert, _ = CELLS[("V1", True)]
print("\nV1 ratio by tertile (rows) and relative orientation 0, 45, 90 (pooled 45):")
for t_ in ["low", "medium", "high"]:
    print(t_, np.round(np.exp(np.log(R[tert == t_]).mean(0)), 3), "units", int((tert == t_).sum()))
print("\nother cortical areas, main effect and interaction (log, pooled 45):")
print(anova_table(["LM", "AL", "RL", "PM", "AM"]).query("scale == 'log' and level45 == 'pooled (c)'").round(3).to_string())

**Result: reproduced.** In V1 the main effect of relative orientation is significant under all four readings (p = 3.1e-09 on log ratios with the 45 degree trials pooled, 9.4e-14 to 2.6e-10 under the others), and the 0 degree ratio is the lowest (0.799 against 0.875 at 45 and 0.927 at 90). 78 of 580 V1 units (96 with averaged 45) are left out for a cell with no spikes, so the ANOVA has 502 units.

## fig2-v1-tuning-interaction

> The dependence of V1 adaptation on relative orientation is strongest in highly
> orientation-selective neurons (interaction of relative orientation and OSI).

In [ ]:
for pool in [True, False]:
    R, tert, _ = CELLS[("V1", pool)]
    for scale in ["log", "ratio"]:
        r = ANOVA[("V1", pool, scale)]["interaction"]
        spread = {t_: float(np.ptp(np.exp(np.log(R[tert == t_]).mean(0)))) for t_ in ["low", "medium", "high"]}
        print(f"45 {'pooled' if pool else 'averaged'}, {scale}: interaction F = {r[0]:.2f}, p = {r[1]:.2g}; "
              f"spread of ratios over relative orientation by tertile {np.round(list(spread.values()), 3)}; "
              f"largest in high: {max(spread, key=spread.get) == 'high'}")

**Result: reproduced.** The interaction of relative orientation and OSI tertile has p = 0.011 (log, pooled 45; 6.3e-07 to 0.018 under the other readings), and the spread of ratios over relative orientation grows from 0.057 (low) to 0.125 (medium) to 0.248 (high OSI).

## fig2-v1-adaptation-at-all-orientations

> V1 neurons adapt for all adaptor orientations relative to their preference, except
> nonpreferred (90 degree) adaptor and test in the most selective units.

Per cell of tertile and relative orientation, a t test of log ratios against zero, corrected by
Benjamini-Hochberg (O6) over (a) the nine V1 cells or (b) all 72 cells of the eight areas;
Bonferroni as a variant.

In [ ]:
def cell_tests(pool):
    rows = []
    for region in A.CORTEX + A.THALAMUS:
        R, tert, _ = CELLS[(region, pool)]
        for t_ in ["low", "medium", "high"]:
            for j, rel in enumerate([0, 45, 90]):
                x = np.log(R[tert == t_, j])
                tt = stats.ttest_1samp(x, 0)
                rows.append(dict(region=region, tertile=t_, rel=rel, n=len(x), ratio=np.exp(x.mean()),
                                 t=tt.statistic, p=tt.pvalue))
    df = pd.DataFrame(rows)
    df["p_bh_all"] = A.fdr(df.p.values)[0]
    df["p_bonf_all"] = np.minimum(df.p * len(df), 1)
    v = df.region == "V1"
    df.loc[v, "p_bh_v1"] = A.fdr(df.p[v].values)[0]
    df.loc[v, "p_bonf_v1"] = np.minimum(df.p[v] * 9, 1)
    return df
for pool in [True, False]:
    ct = cell_tests(pool)
    v = ct[ct.region == "V1"].copy()
    print(f"45 {'pooled (c)' if pool else 'averaged (d)'}")
    show = v.copy()
    for c in ["p", "p_bh_all", "p_bonf_all", "p_bh_v1", "p_bonf_v1"]:
        show[c] = show[c].map(lambda x: f"{x:.2g}")
    print(show.round(3).to_string())
    for col in ["p_bh_v1", "p_bh_all", "p_bonf_v1", "p_bonf_all"]:
        other = v[~((v.tertile == "high") & (v.rel == 90))]
        hi90 = v[(v.tertile == "high") & (v.rel == 90)].iloc[0]
        n_ok = int(((other[col] < 0.05) & (other.ratio < 1)).sum())
        print(f"  {col}: {n_ok} of 8 other cells below 1 at corrected p < 0.05; high OSI at 90: "
              f"ratio {hi90.ratio:.3f}, corrected p {hi90[col]:.2g}")

In [ ]:
# Units left out of Figure 2's analysis, by OSI tertile and the relative orientation whose ratio
# is not finite in log (a zero repeat or orthogonal rate), with 45 degrees pooled.
with np.errstate(divide="ignore", invalid="ignore"):
    out_v1, osi_v1, tert_v1 = tuning_cells(DG, "V1")
    R_all = out_v1[True]
    bad = ~np.isfinite(np.log(R_all))
rows = []
for t_ in ["low", "medium", "high"]:
    m = tert_v1 == t_
    rows.append(dict(tertile=t_, units=int(m.sum()), kept=int((m & ~bad.any(1)).sum()),
                     **{f"not_finite_{rel}": int(bad[m, j].sum()) for j, rel in enumerate([0, 45, 90])},
                     **{f"repeat_zero_{rel}": int((m & (R_all[:, j] == 0)).sum()) for j, rel in enumerate([0, 45, 90])}))
print("V1, 45 pooled:"); print(pd.DataFrame(rows).to_string(index=False))
# The high-OSI 90 degree cell on the ratio scale, units with a zero repeat rate kept (ratio 0);
# a zero orthogonal rate gives no ratio and is left out.
hi = (tert_v1 == "high")
x90 = R_all[hi, 2]
fin = np.isfinite(x90)
tt = stats.ttest_1samp(x90[fin], 1.0)
print(f"high OSI, 90 degrees, ratio scale with zero-repeat units kept: {int(fin.sum())} units "
      f"({int((x90[fin] == 0).sum())} at 0, {int((~fin).sum())} with no ratio), mean ratio {x90[fin].mean():.3f}, "
      f"t = {tt.statistic:.2f}, p = {tt.pvalue:.2g}")
x90k = R_all[hi & ~bad.any(1), 2]
tt = stats.ttest_1samp(np.log(x90k), 0)
print(f"high OSI, 90 degrees, as analysed (log, units with any zero cell left out): {len(x90k)} units, "
      f"ratio {np.exp(np.log(x90k).mean()):.3f}, p = {tt.pvalue:.2g}")
# Variant: tertiles formed after the units with a non-finite cell are left out, so that they are
# equal in the analysis.
keep = ~bad.any(1)
tert2 = np.asarray(pd.qcut(pd.Series(osi_v1[keep]).rank(method="first"), 3, labels=["low", "medium", "high"]).values)
R2 = R_all[keep]
for scale in ["log", "ratio"]:
    Y = np.log(R2) if scale == "log" else R2
    an = A.mixed_anova(Y, tert2)
    print(f"tertiles after exclusion, {scale}: main effect p = {an['within'][1]:.2g}, interaction p = {an['interaction'][1]:.2g}")
rows = []
for t_ in ["low", "medium", "high"]:
    for j, rel in enumerate([0, 45, 90]):
        x = np.log(R2[tert2 == t_, j]); tt = stats.ttest_1samp(x, 0)
        rows.append(dict(tertile=t_, rel=rel, n=len(x), ratio=np.exp(x.mean()), p=tt.pvalue))
c2 = pd.DataFrame(rows); c2["p_bh_v1"] = A.fdr(c2.p.values)[0]
print(c2.round(4).to_string(index=False))
other = c2[~((c2.tertile == "high") & (c2.rel == 90))]
print("  other cells below 1 at BH p < 0.05:", int(((other.p_bh_v1 < 0.05) & (other.ratio < 1)).sum()), "of 8; high 90 BH p",
      round(float(c2[(c2.tertile == "high") & (c2.rel == 90)].p_bh_v1.iloc[0]), 3))
spread = {t_: float(np.ptp(c2[c2.tertile == t_].ratio)) for t_ in ["low", "medium", "high"]}
print("  spread over relative orientation by tertile", {k: round(v, 3) for k, v in spread.items()})


**Result: reproduced.** With the 45 degree trials pooled and Benjamini-Hochberg over the nine V1 cells, all eight other cells are below 1 at corrected p < 0.05 (largest 0.012, medium OSI at 90 degrees), and the high OSI cell at 90 degrees is not (ratio 0.993, p = 0.92). The same holds with the family of all 72 cells. With Bonferroni 7 (V1 family) or 6 (all cells) of 8 pass, and with the 45 degree ratios averaged 7 of 8 pass under Benjamini-Hochberg. The units left out of Figure 2's analysis are almost all in the high OSI tertile (77 of 193, for a zero rate at 45 or 90 degrees; 1 of 193 in the medium tertile, none in the low), so that tertile has 116 units. With tertiles formed after the exclusion (168, 167, 167 units) all eight other cells are below 1 at corrected p < 0.05 and the high OSI cell at 90 degrees is not (0.973, p = 0.62); on the ratio scale, keeping every high OSI unit with a 90 degree ratio, 18 of them 0 (no repeat spikes), gives a mean ratio of 1.116 over 141 units, p = 0.17 against 1 (52 units have no orthogonal spikes and so no ratio). The main effect (p = 3.3e-09) and the interaction (p = 0.033 on log ratios) also stay significant with the re-formed tertiles, and the spread over relative orientation is still largest in the high tertile (0.212, against 0.070 and 0.101).

## fig2-thalamus-no-tuning-dependence

> In LGN and LP, 1-back adaptation does not depend on the adaptor and test orientation relative
> to preference, nor does that dependence interact with orientation selectivity.

A claim of absence, judged on the paper's test: the 3 x 3 mixed ANOVA at alpha 0.05, in the main
reading of the V1 claim (log ratios, pooled 45 degrees); the other readings are shown.

In [ ]:
print(anova_table(A.THALAMUS).round(3).to_string())
for region in A.THALAMUS:
    R, tert, _ = CELLS[(region, True)]
    for t_ in ["low", "medium", "high"]:
        print(region, t_, "ratios at 0, 45, 90:", np.round(np.exp(np.log(R[tert == t_]).mean(0)), 3),
              "units", int((tert == t_).sum()))

**Result: reproduced.** On the paper's test (log ratios, pooled 45 degrees) neither LGN (main effect F = 2.17, p = 0.12; interaction F = 1.37, p = 0.25) nor LP (F = 0.10, p = 0.91; F = 0.19, p = 0.95) shows an effect at 0.05. No reading reaches 0.05; the smallest p is 0.073 (LGN main effect, averaged 45, log). The LGN ratios per cell run from 0.879 to 0.991 and LP from 0.795 to 0.863.

## fig3a-v1-long-term

> V1 neurons adapt significantly to drifting gratings seen up to eight presentations (22 s)
> earlier, despite intervening stimuli.

t tests of unit log ratios against zero for n = 1 to 10, Benjamini-Hochberg at 0.05 per region
(Methods). Every reading of O1 and O3 is run.

In [ ]:
def nback_tests(ar, area, region, n_range):
    x = ar[area == region]
    ps, ts, ms = [], [], []
    for n in n_range:
        t_, p, k = A.ttest_log(x[:, n - 1])
        ps.append(p); ts.append(t_)
        lr = np.log(x[:, n - 1]); ms.append(np.exp(lr[np.isfinite(lr)].mean()))
    adj, rej = A.fdr(ps)
    sig = [n for n, r_, m in zip(n_range, rej, ms) if r_ and m < 1]
    return pd.DataFrame(dict(n=list(n_range), ratio=ms, t=ts, p=ps, p_fdr=adj, sig=rej)), sig

def readings_sig(stim, region, n_range):
    rows = []
    for (st, bl, bk), d in D.items():
        if st != stim:
            continue
        for how in ["pooled", "per_ori", "per_ori_geo"]:
            _, sig = nback_tests(AR[(st, bl, bk)][how], d["area"], region, n_range)
            rows.append(dict(O1=how, blanks=bl, blocks=bk, significant=sig,
                             largest=max(sig) if sig else 0))
    return pd.DataFrame(rows)

In [ ]:
tv1, sig_v1 = nback_tests(ar_dg, DG["area"], "V1", range(1, 11))
show = tv1.copy(); show[["p", "p_fdr"]] = show[["p", "p_fdr"]].map(lambda x: f"{x:.2g}")
print(show.round(4).to_string())
print("main reading: significant n", sig_v1, "largest", max(sig_v1))
rs_v1 = readings_sig("dg", "V1", range(1, 11))
print(rs_v1.to_string())
# By session: the number of mice whose V1 units' mean 8-back log ratio is below zero.
lr8 = np.log(ar_dg[DG["area"] == "V1", 7])
by = pd.Series(lr8).groupby(DG["sess"][DG["area"] == "V1"]).mean()
print(f"8-back: {int((by < 0).sum())} of {len(by)} mice with mean V1 log ratio below zero")

**Result: reproduced.** Under the main reading V1 adaptation is significant at every n from 1 to 8 and not at 9 or 10 (8-back ratio 0.984), so the largest significant n is 8. Seven of 12 readings give 8; the four per-orientation arithmetic readings stop at 2, and one geometric reading at 5. In 22 of 32 mice the mean V1 8-back log ratio is below zero.

## fig3a-v1-double-exponential

> The decay of V1 adaptation over n-back trials is captured by a double exponential with a fast
> (13.99%, 0.85 trials) and a slow (3.45%, 6.82 trials) component.

Least squares on log ratios against the log of the model, n = 1 to 50, bounds of the Methods
(O5). The fit to the region's mean log ratio (O5a) and to every unit's log ratio (O5b) have the
same estimates here, as every unit has a finite value at every n; they differ only in the F test
below. Sign reading (a) both terms reduce the response; (b) as printed.

In [ ]:
def mean_log(ar, area, region):
    lr = np.log(ar[area == region])
    return lr, lr.mean(0)

def fits_for(ar, area, regions):
    rows = {}
    for region in regions:
        lr, m = mean_log(ar, area, region)
        for sign in (-1, 1):
            d, s = A.fit_models(m, sign)
            F, p = A.ftest(s[1], d[1], len(m))
            # unit-level F test (O5b): residuals of every unit's log ratio about the model
            n = np.arange(1, len(m) + 1)
            rss_d = float(np.sum((lr - np.log(A.model_double(n, *d[0], sign=sign))) ** 2))
            rss_s = float(np.sum((lr - np.log(A.model_single(n, *s[0]))) ** 2))
            Fu, pu = A.ftest(rss_s, rss_d, lr.size)
            rows[(region, sign)] = dict(region=region, sign="minus (a)" if sign < 0 else "plus (b)",
                                        a_fast=100 * d[0][0], tau_fast=d[0][1], a_slow=100 * d[0][2],
                                        tau_slow=d[0][3], a_single=100 * s[0][0], tau_single=s[0][1],
                                        F=F, p_F=p, F_units=Fu, p_F_units=pu,
                                        double_on_bound=A.on_bound(d[0], A.B_D),
                                        single_on_bound=A.on_bound(s[0], A.B_S))
    return rows

FIT_DG = fits_for(ar_dg, DG["area"], A.CORTEX + A.THALAMUS)
def fit_table(F_, regions):
    df = pd.DataFrame([v for (r, s), v in F_.items() if r in regions])
    for c in ["p_F", "p_F_units"]:
        df[c] = df[c].map(lambda x: f"{x:.2g}")
    return df

In [ ]:
print(fit_table(FIT_DG, ["V1"]).round(3).to_string())
v = FIT_DG[("V1", -1)]
ci = {"a_fast": (10.73, 16.14), "tau_fast": (0.59, 1.13), "a_slow": (1.71, 6.64), "tau_slow": (3.39, 13.5)}
for k_, (lo, hi) in ci.items():
    print(f"{k_}: {v[k_]:.3f} against Table 1 CI [{lo}, {hi}]: inside {lo <= v[k_] <= hi}")
vb = FIT_DG[("V1", 1)]
print(f"sign reading (b): a_slow {vb['a_slow']:.3f}% (the same curve with the slow amplitude's sign flipped)")
fit_rows = []
for (st, bl, bk), d in D.items():
    if st != "dg":
        continue
    for how in ["pooled", "per_ori", "per_ori_geo"]:
        ar = AR[(st, bl, bk)][how]
        lr = np.log(ar[d["area"] == "V1"])
        lr = np.where(np.isfinite(lr), lr, np.nan)
        dd, ss = A.fit_models(np.nanmean(lr, 0), -1)
        inside = all(lo <= x <= hi for x, (lo, hi) in zip(
            [100 * dd[0][0], dd[0][1], 100 * dd[0][2], dd[0][3]], ci.values()))
        fit_rows.append(dict(O1=how, blanks=bl, blocks=bk, a_fast=100 * dd[0][0], tau_fast=dd[0][1],
                             a_slow=100 * dd[0][2], tau_slow=dd[0][3], all_inside=inside))
print(pd.DataFrame(fit_rows).round(3).to_string())

**Result: reproduced.** The fit gives a_fast 14.21%, tau_fast 0.816 trials, a_slow 3.76% and tau_slow 6.71 trials, each inside its Table 1 interval, under sign reading (a). Reading (b) fits the same curve with a_slow = -3.76%, which is outside its interval as a signed value. Eight of 12 readings of O1 and O3 put all four inside; the per-orientation arithmetic readings put tau_slow on its bound of 50.

## fig3a-v1-half-life

> Adaptation in this study has a half-life of about 14 s; read here as the slow V1 component for
> drifting gratings.

tau_slow x 3 s per trial x ln 2 (criterion).

In [ ]:
trial_s = float(np.median(isi))
hl = FIT_DG[("V1", -1)]["tau_slow"] * 3 * np.log(2)
print(f"tau_slow {FIT_DG[('V1', -1)]['tau_slow']:.3f} trials; half-life {hl:.2f} s with 3 s per trial "
      f"(measured median onset spacing {trial_s:.3f} s gives {FIT_DG[('V1', -1)]['tau_slow'] * trial_s * np.log(2):.2f} s)")
print(f"inside 14 +- 1.4 s: {abs(hl - 14) <= 1.4}")

**Result: reproduced.** tau_slow of 6.714 trials at 3 s per trial gives a half-life of 13.96 s, within 14 +- 1.4 s (13.97 s with the measured 3.003 s spacing).

## fig4-cumulative-adaptation

> Taken together, 2- to 8-back stimuli would reduce the V1 response by 19%, more than the 17%
> reduction from the 1-back stimulus, assuming linear accumulation.

In [ ]:
rows = []
for (st, bl, bk), d in D.items():
    if st != "dg":
        continue
    for how in ["pooled", "per_ori", "per_ori_geo"]:
        ar = AR[(st, bl, bk)][how]
        for agg in ["geo", "mean"]:
            rv = [A.region_value(ar[d["area"] == "V1", n], agg) for n in range(8)]
            cum = 100 * (1 - np.prod(rv[1:8])); one = 100 * (1 - rv[0])
            rows.append(dict(O1=how, O2=agg, blanks=bl, blocks=bk, one_back=one, cumulative_2_8=cum,
                             both_within=abs(cum - 19) <= 1.9 and abs(one - 17) <= 1.7,
                             cum_exceeds=cum > one))
cum_tab = pd.DataFrame(rows)
print(cum_tab.round(2).to_string())
print("main reading (row 0): 1-back", round(cum_tab.one_back[0], 2), "% and 2- to 8-back", round(cum_tab.cumulative_2_8[0], 2), "%")

**Result: reproduced.** Under the main reading the 1-back reduction is 17.88% and the cumulative 2- to 8-back reduction 20.36%, both within tolerance, and the cumulative one is larger. The four pooled geometric readings all match (18.89 to 20.36% against 16.80 to 17.88%); the arithmetic and per-orientation readings do not.

## fig4b-linear-accumulation

> When two adjacent past stimuli both had the current orientation, the observed adaptation
> roughly matches that predicted by multiplying the single n-back effects.

Observed: rate when the n-back and (n+1)-back gratings both had the current orientation over
(c) the rate when both were orthogonal, or (d) the n-back orthogonal and the (n+1)-back any
orientation. Predicted: the product of the region's single n-back and (n+1)-back ratios. Pairs
(a) 1-2 or (b) every pair from 1-2 to 7-8. 95% CI of the observed ratio by bootstrap over V1
units (1000 resamples, the claim's seed).

In [ ]:
def pair_ratios(region, blanks="hold", blocks="within"):
    out = {"c": [], "d": []}
    for i, s in enumerate(S):
        r = select(s, "dg", **MAIN)[0]
        area = s["units"].area.values[s["sel"]]
        r &= area == region
        if r.sum() == 0:
            continue
        ori, blk, order = A.sequence(s["dg"], None, blanks, blocks)
        c = s["dg_counts"][r][:, order].astype(float)
        per = {"c": [], "d": []}
        for n in range(1, 8):
            cur = ori[n + 1:]
            p1 = ori[1: len(ori) - n]               # n-back of position i is i - n
            p2 = ori[: len(ori) - n - 1]            # (n+1)-back
            same = (blk[n + 1:] == blk[: len(ori) - n - 1])
            def rel(a, b):
                dlt = np.abs(a - b); return np.minimum(dlt, 180 - dlt)
            v = same & ~np.isnan(cur) & ~np.isnan(p1) & ~np.isnan(p2)
            both_rep = v & (rel(cur, p1) == 0) & (rel(cur, p2) == 0)
            both_orth = v & (rel(cur, p1) == 90) & (rel(cur, p2) == 90)
            n_orth = v & (rel(cur, p1) == 90)
            cc = c[:, n + 1:]
            rep = cc[:, both_rep].mean(1)
            per["c"].append(rep / cc[:, both_orth].mean(1))
            per["d"].append(rep / cc[:, n_orth].mean(1))
        for k in per:
            out[k].append(np.stack(per[k], 1))
    return {k: np.concatenate(v) for k, v in out.items()}

In [ ]:
PR = pair_ratios("V1")
rng = np.random.default_rng(seed("fig4b-linear-accumulation"))
single = [A.region_value(ar_dg[DG["area"] == "V1", n], "geo") for n in range(8)]
def boot_ci(lr, rng, n_boot=1000):
    U = len(lr)
    b = np.array([lr[rng.integers(0, U, U)].mean() for _ in range(n_boot)])
    return np.exp(np.percentile(b, [2.5, 97.5]))
def fig4b_eval(rng, verbose=True):
    res = {}
    for base in ["c", "d"]:
        lr = np.log(PR[base])
        rows = []
        for n in range(1, 8):
            x = lr[:, n - 1]; ok = np.isfinite(x)
            lo, hi = boot_ci(x[ok], rng)
            pred = single[n - 1] * single[n]
            rows.append(dict(pair=f"{n}-{n + 1}", units=int(ok.sum()), dropped=int((~ok).sum()),
                             observed=np.exp(x[ok].mean()), ci_low=lo, ci_high=hi, predicted=pred,
                             inside=lo <= pred <= hi))
        res[base] = pd.DataFrame(rows)
    return res
F4 = fig4b_eval(rng)
for base, df in F4.items():
    print(f"baseline ({base})"); print(df.round(4).to_string())
    print(f"  (a) pair 1-2 inside: {bool(df.inside.iloc[0])}; (b) all pairs inside: {bool(df.inside.all())} "
          f"({int(df.inside.sum())} of 7)")
for k in [1000, 2000, 3000, 4000]:
    alt = fig4b_eval(np.random.default_rng(seed("fig4b-linear-accumulation") + k))
    print(f"seed + {k}: inside per pair, baseline (c) {alt['c'].inside.astype(int).tolist()}, "
          f"(d) {alt['d'].inside.astype(int).tolist()}")

**Result: reproduced.** With the both-orthogonal baseline (c) the predicted ratio lies inside the bootstrapped 95% interval of the observed one for all seven pairs from 1-2 to 7-8 (1-2: observed 0.765, interval 0.738 to 0.793, predicted 0.754), so readings (a) and (b) both hold, under all five seeds. With baseline (d) six of seven pairs hold and pair 1-2 does not (0.788, 0.765 to 0.811, predicted 0.754), under all five seeds.

## fig3b-thalamus-short-term

> Adaptation to drifting gratings in the thalamus is limited to the 1-back trial in LGN and the
> 2-back trial in LP, while cortical areas stay significant for about 6 to 8 trials.

In [ ]:
rows = []
for region in A.CORTEX + A.THALAMUS:
    df, sig = nback_tests(ar_dg, DG["area"], region, range(1, 11))
    rows.append(dict(region=region, significant=sig, largest=max(sig) if sig else 0,
                     ratios_1_10=np.round(df.ratio.values, 3).tolist()))
sig_tab = pd.DataFrame(rows)
print(sig_tab.to_string())
for region in A.THALAMUS + ["RL"]:
    print(region, "under every reading of O1 and O3:")
    print(readings_sig("dg", region, range(1, 11)).to_string())
# Significance regardless of direction (a two-tailed test can also find a ratio above 1).
for region in A.THALAMUS:
    df, _ = nback_tests(ar_dg, DG["area"], region, range(1, 11))
    print(region, "significant in either direction:", df.n[df.sig].tolist(),
          "ratios there", np.round(df.ratio[df.sig].values, 3).tolist())
rows = []
for (st, bl, bk), d in D.items():
    if st != "dg":
        continue
    for how in ["pooled", "per_ori", "per_ori_geo"]:
        ar = AR[(st, bl, bk)][how]
        lg = {}
        for region in A.CORTEX + A.THALAMUS:
            df, sig = nback_tests(ar, d["area"], region, range(1, 11))
            lg[region] = max(sig) if sig else 0
            lg[region + "_any"] = df.n[df.sig].tolist()
        ok = (lg["LGN"] <= 1 and lg["LP"] <= 2 and all(lg[c] >= 6 for c in A.CORTEX))
        rows.append(dict(O1=how, blanks=bl, blocks=bk, **{k: v for k, v in lg.items() if not k.endswith("_any")},
                         LGN_any=lg["LGN_any"], LP_any=lg["LP_any"], criterion_met=ok))
print(pd.DataFrame(rows).to_string())

In [ ]:
# Two-sided reading: an n counts when its corrected p < 0.05 in either direction. Per reading of
# O1 and O3, the largest such n per area, the n with a ratio above 1, and the criterion and its
# partial rule under this reading.
rows = []
for (st, bl, bk), d in D.items():
    if st != "dg":
        continue
    for how in ["pooled", "per_ori", "per_ori_geo"]:
        ar = AR[(st, bl, bk)][how]
        two, above = {}, {}
        for region in A.CORTEX + A.THALAMUS:
            df, _ = nback_tests(ar, d["area"], region, range(1, 11))
            s2 = df.n[df.sig].tolist()
            two[region] = max(s2) if s2 else 0
            above[region] = df.n[df.sig & (df.ratio > 1)].tolist()
        ok2 = two["LGN"] <= 1 and two["LP"] <= 2 and all(two[c] >= 6 for c in A.CORTEX)
        def later(region, limit):
            df, _ = nback_tests(ar, d["area"], region, range(1, 11))
            return [n for n in df.n[df.sig] if n > limit]
        extra = later("LGN", 1) + later("LP", 2)
        part2 = (not ok2) and len(extra) == 1 and all(two[c] >= 6 for c in A.CORTEX)
        rows.append(dict(O1=how, blanks=bl, blocks=bk, **{f"{r}_2s": two[r] for r in A.CORTEX + A.THALAMUS},
                         above_1={r: v for r, v in above.items() if v}, met_2s=ok2, partial_2s=part2))
print(pd.DataFrame(rows).to_string())
df, _ = nback_tests(AR[("dg", "hold", "across")]["pooled"], D[("dg", "hold", "across")]["area"], "LGN", range(1, 11))
print("LGN, pooled ratio, blanks held, pairs across blocks:")
print(df.assign(p=df.p.map(lambda x: f"{x:.2g}"), p_fdr=df.p_fdr.map(lambda x: f"{x:.2g}")).round(4).to_string(index=False))
# The same with `noise` units kept (O4 a, c, e otherwise), for every reading of O1 and O3: the
# largest n with significant adaptation (ratio below 1) and in either direction, per area.
rows = []
for bl, bk in itertools.product(["hold", "remove"], ["within", "across"]):
    dn = build("dg", sel_kw=NOISE_KW, blanks=bl, blocks=bk)
    for how in ["pooled", "per_ori", "per_ori_geo"]:
        ar = A.ratios(dn["nb"], how)
        one, two = {}, {}
        for region in A.CORTEX + A.THALAMUS:
            df, sig = nback_tests(ar, dn["area"], region, range(1, 11))
            one[region] = max(sig) if sig else 0
            s2 = df.n[df.sig].tolist(); two[region] = max(s2) if s2 else 0
        ok1 = one["LGN"] <= 1 and one["LP"] <= 2 and all(one[c] >= 6 for c in A.CORTEX)
        ok2 = two["LGN"] <= 1 and two["LP"] <= 2 and all(two[c] >= 6 for c in A.CORTEX)
        rows.append(dict(O1=how, blanks=bl, blocks=bk, **one, LGN_2s=two["LGN"], LP_2s=two["LP"],
                         met=ok1, met_2s=ok2))
print("noise units kept:")
print(pd.DataFrame(rows).to_string())


In [ ]:
# Variant: presentations that overlap an invalid interval of their session left out, as current
# trials and from the responsiveness test (they still count as the n-back stimulus). All units of
# the session are treated so, whichever probe the interval names: a superset of a per-probe rule.
def invalid_mask(s, stim):
    t, inv = s[stim], s["invalid"]
    bad = np.zeros(len(t), bool)
    for a, b in zip(getattr(inv, "start_time", []), getattr(inv, "stop_time", [])):
        bad |= (t.start_time.values < b) & (t.stop_time.values > a)
    return ~bad

def build_valid(stim, blocks="within"):
    out = dict(area=[], nb=[])
    for i, s in enumerate(S):
        ok = invalid_mask(s, stim)
        c = s["dg_counts"] if stim == "dg" else s["sg_counts"]
        sp = s["sp100"] if stim == "dg" else s["sp250"]
        w = A.DG_WIN if stim == "dg" else A.SG_WIN
        table = s[stim]
        _, pr, _, _ = A.preferred(c[:, ok], table[ok], w, "orientation")
        gm, gs = A.gray_stats(c[:, ok], table[ok], sp, w, "blank")
        r = A.responsive(pr, gm, gs, 5.0, "mean_plus_sd") & (s["units"].quality.values[s["sel"]] == "good")
        area = s["units"].area.values[s["sel"]]
        r &= pd.notna(area)
        nb, _ = A.nback_rates(c[r], table, 50, "hold", blocks, valid=ok)
        out["area"].append(area[r]); out["nb"].append(nb)
    out["area"] = np.concatenate(out["area"])
    out["nb"] = {k: np.concatenate([nb[k] for nb in out["nb"]]) for k in ["rs", "rn", "os", "on"]}
    return out

for stim, base, ar_b in [("dg", DG, ar_dg), ("sg", SG, ar_sg)]:
    n_bad = [int((~invalid_mask(s, stim)).sum()) for s in S]
    print(f"{'drifting' if stim == 'dg' else 'static'}: presentations in invalid intervals, "
          f"{sum(n_bad)} in {sum(x > 0 for x in n_bad)} sessions (largest {max(n_bad)} in one session)")
    dv = build_valid(stim)
    arv_ = A.ratios(dv["nb"], "pooled")
    rows = []
    for a in A.CORTEX + A.THALAMUS:
        rows.append(dict(area=a, units=int((base["area"] == a).sum()), units_valid=int((dv["area"] == a).sum()),
                         ratio1=round(A.region_value(ar_b[base["area"] == a, 0]), 4),
                         ratio1_valid=round(A.region_value(arv_[dv["area"] == a, 0]), 4)))
    print(pd.DataFrame(rows).to_string(index=False))
    print("  total responsive units", len(base["area"]), "->", len(dv["area"]))
    n_fam = 10 if stim == "dg" else 20
    for region in ["V1", "LGN", "LP", "RL"]:
        _, sig_b = nback_tests(ar_b, base["area"], region, range(1, n_fam + 1))
        _, sig_v = nback_tests(arv_, dv["area"], region, range(1, n_fam + 1))
        print(f"  {region}: significant n (family 1 to {n_fam}) {sig_b} -> {sig_v}")

# fig3b under this variant, pairs within blocks (main) and across blocks.
for blocks in ["within", "across"]:
    dv = build_valid("dg", blocks)
    arv_ = A.ratios(dv["nb"], "pooled")
    lg = {}
    for region in A.CORTEX + A.THALAMUS:
        _, sig = nback_tests(arv_, dv["area"], region, range(1, 11))
        lg[region] = max(sig) if sig else 0
    ok = lg["LGN"] <= 1 and lg["LP"] <= 2 and all(lg[c] >= 6 for c in A.CORTEX)
    print(f"fig3b, invalid presentations left out, pairs {blocks} blocks: largest significant n {lg}; criterion met {ok}")


**Result: reproduced, resting on an assumption.** Counting significant adaptation (a corrected p < 0.05 with the ratio below 1), the reading with pairs formed across blocks (O1a, O3a, d) meets the criterion: LGN only at 1-back, LP to 2-back, and every cortical area to 6-back or further (V1 8, LM 8, AL 7, RL 6, PM 7, AM 6). RL is at the threshold of 6. Under the main reading (pairs within blocks) RL stops at 5-back and the criterion fails; no other of the 12 readings meets it. Counting significance in either direction (the two-sided table), no reading meets the criterion: under the across-blocks reading LGN is also significant at 9-back (ratio 1.024, corrected p = 0.049), so only the partial rule holds there, and under the main reading LP is significant at 3-back (ratio 1.03) while RL stops at 5. The outcome therefore rests on reading Figure 3B's black borders as marking adaptation, ratios below 1, which the legend's "exerted significant adaptation effect" for panel A and the Results' "adaptation ... appeared to be limited" support; read two-sided, it would be partially reproduced. With the presentations inside invalid intervals left out, the across-blocks reading still meets the criterion (LGN 1, LP 1, RL 6), and under the main reading LGN gains a significant 7-back. With `noise` units kept, no reading of O1 and O3 meets the criterion or its partial rule: under the across-blocks reading RL stops at 5-back (LGN 1, LP 2), and under the main reading LGN is significant at 7-back. The outcome therefore also rests on the `noise` filter.

## v1-vs-thalamus-welch

> Adaptation to remote stimuli is stronger in V1 than in LGN (2-, 4- and 9-back) and LP
> (3-back), even after normalizing to each region's 1-back adaptation.

Per n, a two-sided Welch t test of unit values in V1 against LGN and against LP. Values: the
unnormalized log ratio, (a) each unit's n-back log ratio divided by its own 1-back log ratio,
(b) divided by its region's mean 1-back log ratio. Normalized values are signed so that larger
means more adaptation; for the unnormalized log ratio, more negative is more adaptation. n in
(c) 2 to 10 or (d) 2 to 50, Benjamini-Hochberg within each pair of regions (O6).

In [ ]:
lr_dg = np.log(ar_dg)
def welch(region, norm, n_max):
    a = lr_dg[DG["area"] == "V1"]; b = lr_dg[DG["area"] == region]
    if norm == "unit":
        a = a / a[:, [0]]; b = b / b[:, [0]]
    elif norm == "region":
        a = a / a[:, 0].mean(); b = b / b[:, 0].mean()
    else:
        a, b = -a, -b
    ns = list(range(2, n_max + 1))
    res = [stats.ttest_ind(a[:, n - 1], b[:, n - 1], equal_var=False) for n in ns]
    t_ = np.array([r.statistic for r in res]); p = np.array([r.pvalue for r in res])
    adj = A.fdr(p)[0]
    stronger = [n for n, tt, q in zip(ns, t_, adj) if q < 0.05 and tt > 0]
    weaker = [n for n, tt, q in zip(ns, t_, adj) if q < 0.05 and tt < 0]
    return pd.DataFrame(dict(n=ns, t=t_, p=p, p_fdr=adj)), stronger, weaker
rows = []
for norm in ["none", "unit", "region"]:
    for n_max in [10, 50]:
        _, sL, wL = welch("LGN", norm, n_max)
        _, sP, wP = welch("LP", norm, n_max)
        hits = [n in sL for n in (2, 4, 9)] + [3 in sP]
        rows.append(dict(normalization=norm, n_range=f"2-{n_max}", V1_stronger_than_LGN=sL,
                         V1_stronger_than_LP=sP, V1_weaker_LGN=wL, V1_weaker_LP=wP,
                         of_four=int(sum(hits))))
print(pd.DataFrame(rows).to_string())
df, _, _ = welch("LGN", "region", 10)
print("\nV1 against LGN, normalized by region (b), n = 2 to 10")
print(df.assign(p=df.p.map(lambda x: f"{x:.2g}"), p_fdr=df.p_fdr.map(lambda x: f"{x:.2g}")).round(3).to_string())
df, _, _ = welch("LP", "region", 10)
print("\nV1 against LP, normalized by region (b), n = 2 to 10")
print(df.assign(p=df.p.map(lambda x: f"{x:.2g}"), p_fdr=df.p_fdr.map(lambda x: f"{x:.2g}")).round(3).to_string())
u1 = lr_dg[:, 0]
print(f"\nunit 1-back log ratios within 0.02 of zero (unstable under (a)): "
      f"{int((np.abs(u1[DG['area'] == 'V1']) < 0.02).sum())} V1, {int((np.abs(u1[DG['area'] == 'LGN']) < 0.02).sum())} LGN, "
      f"{int((np.abs(u1[DG['area'] == 'LP']) < 0.02).sum())} LP")

**Result: reproduced.** With n = 2 to 10 and correction within each pair of regions, V1 adaptation is stronger than LGN at exactly 2, 4 and 9 back and than LP at 3 back, both without normalization (and also LP at 4) and with each region's 1-back mean as the normalization (b). Normalizing each unit by its own 1-back log ratio (a) finds no difference; 38 V1, 11 LGN and 8 LP units have a 1-back log ratio within 0.02 of zero, which makes that measure unstable. Over n = 2 to 50, 3 of the four remain unnormalized and 1 with reading (b).

## fig3c-mice-long-term

> Long-term (4- to 8-back) adaptation is consistent across mice in cortex but not in thalamus.

In [ ]:
for pooled_i, lab in [(0, "trials pooled over 4 to 8"), (1, "rates averaged over 4 to 8")]:
    gC = mouse_rates(DG, A.CORTEX, range(4, 9), A.DG_WIN)[pooled_i]
    gT = mouse_rates(DG, A.THALAMUS, range(4, 9), A.DG_WIN)[pooled_i]
    print(lab)
    ca = consistency(gC, "  cortex")
    ta = consistency(gT, "  thalamus")
    print(f"  cortex meets (a) or (b): {ca[0] or ca[1]}; thalamus meets neither: {not (ta[0] or ta[1])}")

**Result: reproduced.** For 4- to 8-back, 31 of 32 mice (pooled trials; 30 with averaged rates) have the cortical orthogonal rate above the repeat rate (sign test p = 1.5e-08), so cortex meets (b). In thalamus 14 of 23 mice do (p = 0.4), so neither reading holds, as the claim expects.

## decay-cortex-double-better

> Recovery from adaptation to drifting gratings in cortical areas is better fit by a double than
> a single exponential, in all areas but RL.

F tests of the nested fits (O5): on the region's mean log ratio over n = 1 to 50 (O5a, 50
observations) and on every unit's log ratio (O5b).

In [ ]:
ft = fit_table(FIT_DG, A.CORTEX)
print(ft[ft.sign == "minus (a)"].round(3).to_string())
for col in ["p_F", "p_F_units"]:
    pv = {r: float(FIT_DG[(r, -1)][col]) for r in A.CORTEX}
    print(f"{col}: below 0.05 in V1, LM, AL, PM, AM: {all(pv[r] < 0.05 for r in ['V1', 'LM', 'AL', 'PM', 'AM'])}; "
          f"RL p = {pv['RL']:.2g}")

**Result: reproduced.** The F test favours the double exponential in V1 (p = 0.0015), LM (0.017), AL (8.0e-05), PM (0.0041) and AM (0.011) on the region means, and in all five also with every unit's values (p = 4.5e-09 to 0.0021). RL is not significant (p = 0.22, units 0.12), as in the paper (0.072). Two of the five p values (LM, AM) are above the paper's bound of 0.006.

## table1-extrastriate-tau-slow

> Recovery from adaptation to drifting gratings is slowest in V1 (tau_slow 6.82 trials), and
> relatively similar in extrastriate areas, whose slow time constants range from 3.12 to 5.52
> trials with all 95% CIs overlapping.

Bootstrap of the double-exponential fit (O5): units resampled with replacement 1000 times, fits
on a bound discarded, 2.5 and 97.5 percentiles (Methods). The claim's seed, and four more for
the seed's effect.

In [ ]:
T1 = {"V1": (3.39, 13.5), "AL": (3.1, 5.94), "AM": (2.37, 6.5), "LM": (3.42, 10.6),
      "PM": (2.75, 9.28), "RL": (2.19, 34.41)}
T1_other = {"V1": [(10.73, 16.14), (0.59, 1.13), (1.71, 6.64)], "AL": [(8.14, 12.71), (1e-3, 0.47), (4.90, 8.54)],
            "AM": [(6.88, 14.30), (5e-3, 0.87), (3.83, 11.04)], "LM": [(10.37, 16.07), (0.02, 0.72), (1.89, 5.84)],
            "PM": [(10.43, 16.32), (2e-3, 0.45), (3.91, 9.26)], "RL": [(1.97, 12.00), (5e-3, 1.83), (0.7, 9.82)]}
t = time.time()
base_seed = seed("table1-extrastriate-tau-slow")
tasks = [(np.log(ar_dg[DG["area"] == r]), "double", base_seed + k) for k in [0, 1000, 2000, 3000, 4000] for r in A.CORTEX]
BOOT_DG = dict(zip([(k, r) for k in [0, 1000, 2000, 3000, 4000] for r in A.CORTEX], A.bootstrap_many(tasks)))
print(f"bootstraps in {time.time() - t:.0f} s")
def overlap_all(cis):
    lo = max(c[0] for c in cis.values()); hi = min(c[1] for c in cis.values())
    return lo <= hi
for k in [0, 1000, 2000, 3000, 4000]:
    cis = {r: tuple(np.percentile(BOOT_DG[(k, r)][0][:, 3], [2.5, 97.5])) for r in A.CORTEX}
    if k == 0:
        rows = []
        for r in A.CORTEX:
            b, dropped = BOOT_DG[(0, r)]
            ci4 = np.percentile(b, [2.5, 97.5], 0)
            est = FIT_DG[(r, -1)]
            rows.append(dict(region=r, tau_slow=est["tau_slow"], boot_low=ci4[0, 3], boot_high=ci4[1, 3],
                             table1_ci=T1[r], inside=T1[r][0] <= est["tau_slow"] <= T1[r][1],
                             a_fast=est["a_fast"], a_fast_in=T1_other[r][0][0] <= est["a_fast"] <= T1_other[r][0][1],
                             tau_fast=est["tau_fast"], tau_fast_in=T1_other[r][1][0] <= est["tau_fast"] <= T1_other[r][1][1],
                             a_slow=est["a_slow"], a_slow_in=T1_other[r][2][0] <= est["a_slow"] <= T1_other[r][2][1],
                             kept=len(b), dropped=dropped))
        t1 = pd.DataFrame(rows)
        print(t1.round(3).to_string())
        ex5 = t1[t1.region != "V1"]
        print(f"extrastriate tau_slow inside Table 1 CI: {int(ex5.inside.sum())} of 5; "
              f"range {ex5.tau_slow.min():.2f} to {ex5.tau_slow.max():.2f}; V1 largest of six: "
              f"{t1.tau_slow.idxmax() == 0}")
    print(f"seed + {k}: all six bootstrapped tau_slow CIs overlap: {overlap_all(cis)}; "
          f"CIs {', '.join(f'{r} [{c[0]:.2f}, {c[1]:.2f}]' for r, c in cis.items())}")

**Result: reproduced.** The five extrastriate tau_slow (AL 3.93, AM 3.23, LM 5.66, PM 4.27, RL 2.83 trials) are each inside their Table 1 intervals, and the six bootstrapped intervals overlap under all five seeds. V1's tau_slow (6.71) is the largest of the six. All a_fast, tau_fast and a_slow estimates are inside their Table 1 intervals too.

## decay-thalamus-single

> Recovery in LGN and LP is best described by a single exponential with very short time
> constants (0.02 and 0.71 trials).

In [ ]:
print(fit_table(FIT_DG, A.THALAMUS).round(4).to_string())
lims = {"LGN": [(0.004, 0.63), (4.04, 8.85)], "LP": [(0.03, 0.93), (12.68, 21.02)]}
for r in A.THALAMUS:
    e = FIT_DG[(r, -1)]
    print(f"{r}: tau {e['tau_single']:.3f} in {lims[r][0]}: {lims[r][0][0] <= e['tau_single'] <= lims[r][0][1]}; "
          f"a {e['a_single']:.2f}% in {lims[r][1]}: {lims[r][1][0] <= e['a_single'] <= lims[r][1][1]}; "
          f"F test p (mean) {e['p_F']:.2g}, (units) {e['p_F_units']:.2g}")
bt = A.bootstrap_many([(np.log(ar_dg[DG["area"] == r]), "single", seed("decay-thalamus-single")) for r in A.THALAMUS])
for r, (b, dropped) in zip(A.THALAMUS, bt):
    print(f"{r} bootstrap (seed of the claim): tau CI [{np.percentile(b[:, 1], 2.5):.3f}, {np.percentile(b[:, 1], 97.5):.3f}], "
          f"a CI [{100 * np.percentile(b[:, 0], 2.5):.2f}, {100 * np.percentile(b[:, 0], 97.5):.2f}]%, dropped {dropped}")

**Result: reproduced, resting on an assumption.** The F test does not favour the double exponential in LGN (p = 0.81) or LP (p = 0.57; 0.62 and 0.19 with unit values), and the single exponential gives LGN tau 0.031 trials and a 6.48%, LP tau 0.712 and a 16.85%, each inside its Table 1 interval. The double-exponential fits reach a bound in both. With `noise` units kept (variant above) the LGN tau falls to 0.000, so the outcome rests on that filter.

## fig5-repeat-suppression

> Against a blank baseline, both repeated and orthogonal 1-back gratings suppress cortical
> responses, but only the suppression by repeated orientations persists, up to eight trials back.

All cortical units pooled. Per unit and n, the rate when the n-back trial was a repeat over the
rate when it was a blank, and orthogonal over blank; repeat and orthogonal trials are subsampled
per session to the number of n-back blank trials (Methods). Blanks hold positions (O3a), pairs
within blocks. Readings of "significant": (a) the bootstrapped 95% CI over units excludes 1;
(b) t tests of log ratios against zero, n = 1 to 10, Benjamini-Hochberg.

In [ ]:
def blank_ratios(rng, n_max=10):
    rep_all, orth_all = [], []
    for i, s in enumerate(S):
        r = select(s, "dg", **MAIN)[0]
        area = s["units"].area.values[s["sel"]]
        r &= np.isin(area, A.CORTEX)
        if r.sum() == 0:
            continue
        ori, blk, order = A.sequence(s["dg"], None, "hold", "within")
        c = s["dg_counts"][r][:, order].astype(float)
        R, O = np.full((r.sum(), n_max), np.nan), np.full((r.sum(), n_max), np.nan)
        for n in range(1, n_max + 1):
            rep, orth, blank = A.pair_masks(ori, blk, n)
            cc = c[:, n:]
            nb_ = int(blank.sum())
            ri = rng.choice(np.flatnonzero(rep), nb_, replace=False)
            oi = rng.choice(np.flatnonzero(orth), nb_, replace=False)
            b = cc[:, blank].mean(1)
            with np.errstate(divide="ignore", invalid="ignore"):
                R[:, n - 1] = cc[:, ri].mean(1) / b
                O[:, n - 1] = cc[:, oi].mean(1) / b
        rep_all.append(R); orth_all.append(O)
    return np.concatenate(rep_all), np.concatenate(orth_all)

def fig5_eval(R, O, rng_boot):
    rows = []
    for name, X in [("repeat/blank", R), ("orthogonal/blank", O)]:
        with np.errstate(divide="ignore", invalid="ignore"):
            lr = np.log(X)
        ps = []
        for n in range(1, X.shape[1] + 1):
            x = lr[:, n - 1]; x = x[np.isfinite(x)]
            lo, hi = boot_ci(x, rng_boot, 1000)
            p = stats.ttest_1samp(x, 0).pvalue; ps.append(p)
            rows.append(dict(ratio=name, n=n, units=len(x), dropped=int(np.sum(~np.isfinite(lr[:, n - 1]))), value=np.exp(x.mean()), ci_low=lo, ci_high=hi, p=p))
    df = pd.DataFrame(rows)
    for name in ["repeat/blank", "orthogonal/blank"]:
        m = df.ratio == name
        df.loc[m, "p_fdr"] = A.fdr(df.p[m].values)[0]
    df["below_ci"] = df.ci_high < 1
    df["below_t"] = (df.p_fdr < 0.05) & (df.value < 1)
    return df

def fig5_criterion(df, col):
    r = df[df.ratio == "repeat/blank"].set_index("n"); o = df[df.ratio == "orthogonal/blank"].set_index("n")
    ok = (r.value[1] < 1 and o.value[1] < 1 and not o[col].loc[2:10].any()
          and bool(r[col][8]) and int(r[col].loc[2:8].sum()) >= 4)
    return ok, r[col].loc[1:10].astype(int).tolist(), o[col].loc[1:10].astype(int).tolist()

In [ ]:
s5 = seed("fig5-repeat-suppression")
blank_counts = [int(A.pair_masks(*A.sequence(s["dg"], None, "hold", "within")[:2], 1)[2].sum()) for s in S]
print("1-back blank trials per session:", min(blank_counts), "to", max(blank_counts))
R5, O5 = blank_ratios(np.random.default_rng(s5))
df5 = fig5_eval(R5, O5, np.random.default_rng(s5 + 1))
show = df5.copy(); show[["p", "p_fdr"]] = show[["p", "p_fdr"]].map(lambda x: f"{x:.2g}")
print(show.round(4).to_string())
for col, lab in [("below_ci", "(a) bootstrap CI"), ("below_t", "(b) t test with FDR")]:
    ok, rr, oo = fig5_criterion(df5, col)
    print(f"{lab}: criterion met {ok}; repeat below 1 at n=1..10 {rr}; orthogonal {oo}")
for k in [1000, 2000, 3000, 4000]:
    Rk, Ok = blank_ratios(np.random.default_rng(s5 + k))
    dk = fig5_eval(Rk, Ok, np.random.default_rng(s5 + k + 1))
    print(f"seed + {k}: (a) {fig5_criterion(dk, 'below_ci')}, (b) {fig5_criterion(dk, 'below_t')}")
# Variant: 100 subsampling draws, the unit ratios averaged in log over draws.
rngv = np.random.default_rng(s5 + 2)
acc_r, acc_o = [], []
for _ in range(100):
    Rk, Ok = blank_ratios(rngv)
    with np.errstate(divide="ignore", invalid="ignore"):
        acc_r.append(np.log(Rk)); acc_o.append(np.log(Ok))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    Rv = np.exp(np.nanmean(np.where(np.isfinite(acc_r), acc_r, np.nan), 0))
    Ov = np.exp(np.nanmean(np.where(np.isfinite(acc_o), acc_o, np.nan), 0))
dv = fig5_eval(Rv, Ov, np.random.default_rng(s5 + 3))
print("100 draws:", "(a)", fig5_criterion(dv, "below_ci"), "(b)", fig5_criterion(dv, "below_t"))
print(dv[["ratio", "n", "value", "ci_low", "ci_high"]].round(4).to_string())

In [ ]:
# Session (cluster) bootstrap: units of one session share one trial subsample, so sessions are
# resampled with replacement, with all their units, 2000 times.
sess5 = np.concatenate([np.full(int((select(s, "dg", **MAIN)[0] & np.isin(s["units"].area.values[s["sel"]], A.CORTEX)).sum()), i)
                        for i, s in enumerate(S)])
assert len(sess5) == len(R5)
def cluster_ci(X, n, rng, n_boot=2000):
    with np.errstate(divide="ignore", invalid="ignore"):
        lr = np.log(X[:, n - 1])
    ok = np.isfinite(lr); lr, ss = lr[ok], sess5[ok]
    ids = np.unique(ss)
    sums = np.array([lr[ss == k].sum() for k in ids]); cnt = np.array([(ss == k).sum() for k in ids])
    b = []
    for _ in range(n_boot):
        j = rng.integers(0, len(ids), len(ids))
        b.append(sums[j].sum() / cnt[j].sum())
    return np.exp(lr.mean()), *np.exp(np.percentile(b, [2.5, 97.5]))
rngc = np.random.default_rng(s5 + 4)
rows = []
for lab, (Rx, Ox) in [("claim's draw", (R5, O5)), ("100 draws", (Rv, Ov))]:
    for name, X in [("repeat/blank", Rx), ("orthogonal/blank", Ox)]:
        for n in range(1, 11):
            v, lo, hi = cluster_ci(X, n, rngc)
            rows.append(dict(subsample=lab, ratio=name, n=n, value=v, ci_low=lo, ci_high=hi, below=hi < 1))
dc = pd.DataFrame(rows)
for lab in ["claim's draw", "100 draws"]:
    x = dc[dc.subsample == lab]
    r = x[x.ratio == "repeat/blank"].set_index("n"); o = x[x.ratio == "orthogonal/blank"].set_index("n")
    ok = (r.value[1] < 1 and o.value[1] < 1 and not o.below.loc[2:10].any() and bool(r.below[8]) and int(r.below.loc[2:8].sum()) >= 4)
    print(f"{lab}, session bootstrap: criterion met {ok}; repeat below 1 at n=1..10 {r.below.astype(int).tolist()}; "
          f"orthogonal {o.below.astype(int).tolist()}")
    print(f"  orthogonal/blank 2-back {o.value[2]:.4f} ({o.ci_low[2]:.4f} to {o.ci_high[2]:.4f}); "
          f"repeat/blank 8-back {r.value[8]:.4f} ({r.ci_low[8]:.4f} to {r.ci_high[8]:.4f})")


**Result: not reproduced.** Against blank trials both 1-back ratios are below 1 (repeat 0.757, orthogonal 0.922), and repeat suppression lasts longer than orthogonal suppression: by the t tests repeat is significant at 1 to 4 and 7, orthogonal at 1 and 2. The criterion requires orthogonal over blank not below 1 at any n from 2 to 10, and it is below 1 at 2-back (0.972) under both readings, under all five seeds, averaged over 100 subsampling draws (0.978), and with the session bootstrap (0.949 to 0.994 for the claim's draw, 0.961 to 0.996 for the 100 draws). Repeat over blank is not significant at 8-back under the claim's seed (0.999); it is with the 100 draws by the unit bootstrap and the t tests, but not by the session bootstrap (0.960 to 1.002). The criterion gives no partial rule, and the part that fails is the claim's central contrast, that orthogonal suppression is limited to the 1-back trial.

## fig6c-static-v1-1back

> With brief back-to-back static gratings, orientation repetition reduces the V1 response over 0
> to 250 ms, a 1-back ratio of 0.90.

Rates and responsiveness over 0 to 250 ms; orthogonal is 90 degrees apart; spatial frequency
and phase pooled; blank sweeps hold positions (O3 with blank sweeps in place of blank trials).

In [ ]:
rows = []
for good in [True, False]:
    for src in ["blank", "spont"]:
        vc = count_regions("sg", rule="mean_plus_sd", src=src, by="orientation", good=good)
        rows.append(dict(good_only=good, gray=src, **{A.AREA_NAMES[k]: int(vc.get(k, 0)) for k in A.AREA_NAMES},
                         other_labels_50=",".join(sorted(vc[(~vc.index.isin(NAMED)) & (vc >= 50)].index))))
print("static responsive units (O4 a, e), by gray source and noise filter")
print(pd.DataFrame(rows).to_string())
print("paper: V1 530 (tolerance 53), LGN 60 (6), LP 16 (1.6)")
r6 = region_stats(ar_sg, SG["area"])
show = r6.copy(); show[["p", "p_fdr"]] = show[["p", "p_fdr"]].map(lambda x: f"{x:.2g}")
print(show.round(4).to_string())
tab6 = onebak_table("sg")
print(tab6.to_string())
print("readings with V1 in [0.89, 0.91]:", int(tab6.V1.between(0.89, 0.91).sum()), "of", len(tab6))

**Result: reproduced.** Over 0 to 250 ms the V1 static 1-back ratio is 0.897 (t interval 0.889 to 0.905) over 545 units, p = 4e-85, inside [0.89, 0.91], and 545 is within 53 of 530. LGN 61 and LP 17 units match the paper's 60 and 16 with the same selection.

## fig6c-static-other-areas-1back

> Extrastriate areas show similar 1-back adaptation to static gratings (0.89 to 0.93), and the
> few responsive LGN and LP neurons also adapt (0.93 and 0.85).

In [ ]:
ex6 = r6[r6.area.isin(["LM", "AL", "RL", "PM", "AM"])]
print(f"extrastriate: {ex6.ratio.min():.3f} to {ex6.ratio.max():.3f}, all in [0.801, 1.023]: "
      f"{bool(ex6.ratio.between(0.801, 1.023).all())}, largest corrected p {ex6.p_fdr.max():.2g}")
for a_, (lo, hi), n_paper in [("LGN", (0.93 - 0.093, 0.98), 60), ("LP", (0.85 - 0.085, 0.96), 16)]:
    row = r6[r6.area == a_].iloc[0]
    print(f"{a_}: ratio {row.ratio:.3f}, widened CI [{lo:.3f}, {hi}], match {lo <= row.ratio <= hi}; "
          f"p {row.p:.2g} (corrected {row.p_fdr:.2g}); units {row.n_units} against {n_paper}")
tab6["ex_min"] = tab6[["LM", "AL", "RL", "PM", "AM"]].min(axis=1)
tab6["ex_max"] = tab6[["LM", "AL", "RL", "PM", "AM"]].max(axis=1)
print(tab6[["O1", "O2", "blanks", "blocks", "ex_min", "ex_max", "LGN", "LP"]].to_string())

**Result: reproduced.** Extrastriate ratios run from 0.891 to 0.930 (largest corrected p 1.7e-09); LGN 0.957 (p = 0.0025) and LP 0.900 (p = 0.0045) are inside their widened intervals, and the counts (61, 17) are within tolerance.

## fig6d-static-consistent-across-mice

> Static grating 1-back adaptation is consistent across mice.

In [ ]:
for grp, name in [(A.THALAMUS, "thalamus"), (A.CORTEX, "cortex")]:
    g = mouse_rates(SG, grp, [1], A.SG_WIN)[0]
    consistency(g, f"static 1-back, {name}")

**Result: partially reproduced.** All 32 mice have the cortical orthogonal rate above the repeat rate (p = 4.7e-10); in thalamus 15 of 17 mice do (p = 0.0023), so only reading (b) holds there.

## fig7a-static-v1-long-term

> After brief static gratings V1 adaptation persists for stimuli up to 20 presentations (5 s)
> back, with a double exponential decay whose slow time constant is 9.12 trials.

FDR family (a) n = 1 to 20 or (b) 1 to 50; fits as for drifting gratings.

In [ ]:
FIT_SG = fits_for(ar_sg, SG["area"], A.CORTEX + A.THALAMUS)
for n_max in [20, 50]:
    df, sig = nback_tests(ar_sg, SG["area"], "V1", range(1, n_max + 1))
    print(f"family 1-{n_max}: significant n {sig}; largest {max(sig)}; 20-back significant {20 in sig}; "
          f"20-back ratio {df.ratio[19]:.4f}, corrected p {df.p_fdr[19]:.2g}")
print(fit_table(FIT_SG, ["V1"]).round(3).to_string())
ci2 = {"a_fast": (7.11, 9.23), "tau_fast": (0.37, 0.71), "a_slow": (1.33, 2.91), "tau_slow": (6.09, 14.82)}
e = FIT_SG[("V1", -1)]
for k_, (lo, hi) in ci2.items():
    print(f"{k_}: {e[k_]:.3f} in Table 2 CI [{lo}, {hi}]: {lo <= e[k_] <= hi}")
rs7 = readings_sig("sg", "V1", range(1, 21))
print("family 1-20 under every reading of O1 and O3:"); print(rs7.to_string())

**Result: reproduced.** With the FDR family n = 1 to 20 (a), V1 adaptation is significant at 20-back (ratio 0.991, corrected p = 9.3e-05), the largest significant n; with 1 to 50 (b) the largest is 49, outside 18 to 22. The fit gives a_fast 8.26%, tau_fast 0.533, a_slow 2.07% and tau_slow 9.47 trials, all inside their Table 2 intervals.

## fig7b-static-cortex-decay

> Extrastriate areas show similar decay after static gratings, with slow time constants from
> 6.54 (AM) to 21.78 trials (PM), all better fit by a double exponential.

In [ ]:
T2 = {"V1": (6.09, 14.82), "AL": (5.24, 39.75), "AM": (3.82, 19.11), "LM": (5.42, 13.36),
      "PM": (13.07, 37.0), "RL": (10.99, 37.51)}
ft2 = fit_table(FIT_SG, A.CORTEX)
print(ft2[ft2.sign == "minus (a)"].round(3).to_string())
t = time.time()
b7 = seed("fig7b-static-cortex-decay")
tasks = [(np.log(ar_sg[SG["area"] == r]), "double", b7 + k) for k in [0, 1000, 2000, 3000, 4000] for r in A.CORTEX]
BOOT_SG = dict(zip([(k, r) for k in [0, 1000, 2000, 3000, 4000] for r in A.CORTEX], A.bootstrap_many(tasks)))
print(f"bootstraps in {time.time() - t:.0f} s")
for k in [0, 1000, 2000, 3000, 4000]:
    cis = {r: tuple(np.percentile(BOOT_SG[(k, r)][0][:, 3], [2.5, 97.5])) for r in A.CORTEX}
    inside = {r: T2[r][0] <= FIT_SG[(r, -1)]["tau_slow"] <= T2[r][1] for r in A.CORTEX}
    print(f"seed + {k}: CIs {', '.join(f'{r} [{c[0]:.2f}, {c[1]:.2f}]' for r, c in cis.items())}; "
          f"all overlap {overlap_all(cis)}; dropped {[BOOT_SG[(k, r)][1] for r in A.CORTEX]}")
print("tau_slow inside Table 2 CI:", inside)
print("F test p (mean):", {r: FIT_SG[(r, -1)]["p_F"] for r in A.CORTEX})
print("F test p (units):", {r: FIT_SG[(r, -1)]["p_F_units"] for r in A.CORTEX})

**Result: reproduced.** Every cortical area favours the double exponential (F test p from 9.5e-12 to 1.4e-05), every extrastriate tau_slow (AL 13.58, AM 6.29, LM 8.24, PM 19.07, RL 19.70) is inside its Table 2 interval, and the six bootstrapped intervals overlap under all five seeds. One p value (AM, 1.4e-05) is above the paper's bound of 1e-5.

## fig7-static-lgn-single

> After static gratings LGN is better described by a single exponential with a shorter time
> constant (2.93 trials), and the cortex and thalamus difference is less pronounced than for
> drifting gratings.

In [ ]:
e = FIT_SG[("LGN", -1)]
print(fit_table(FIT_SG, ["LGN"]).round(4).to_string())
print(f"LGN single: tau {e['tau_single']:.3f} in [1.36, 7.32]: {1.36 <= e['tau_single'] <= 7.32}; "
      f"a {e['a_single']:.2f}% in [1.80, 6.59]: {1.80 <= e['a_single'] <= 6.59}; F test p {e['p_F']:.2g} "
      f"(units {e['p_F_units']:.2g})")
print(f"reported only: LGN tau static {e['tau_single']:.2f} against drifting {FIT_DG[('LGN', -1)]['tau_single']:.3f} trials")

**Result: reproduced.** In LGN the F test does not favour the double exponential (p = 0.20; 0.14 with unit values), and the single exponential gives tau 2.77 trials and a 4.00%, inside their Table 2 intervals. As reported only, the LGN static tau (2.77) is longer than the drifting one (0.031).

## fig7b-static-areas-long-term

> After static gratings, cortical areas show significant adaptation to orientations presented up
> to 20 trials back, whereas in the thalamus long-term adaptation is less evident.

In [ ]:
rows = []
for n_max in [20, 50]:
    lg = {}
    for region in A.CORTEX + A.THALAMUS:
        df, sig = nback_tests(ar_sg, SG["area"], region, range(1, n_max + 1))
        lg[region] = max(sig) if sig else 0
        rows.append(dict(family=f"1-{n_max}", region=region, largest=lg[region], significant=sig))
    cort = [lg[r] for r in A.CORTEX]
    some = any(18 <= x <= 22 for x in cort); every = all(18 <= x <= 22 for x in cort)
    thal = all(lg[t_] < min(cort) for t_ in A.THALAMUS)
    print(f"family 1-{n_max}: cortical largest {dict(zip(A.CORTEX, cort))}; thalamic {lg['LGN']}, {lg['LP']}; "
          f"(c) some area in 18-22 {some}; (d) every area {every}; thalamus below every cortical area {thal}")
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** With the family 1 to 20, V1, AL, PM and AM are significant at 20-back and LM and RL stop at 15 and 17, so reading (c), some area in 18 to 22, holds and (d), every area, does not. LGN (largest 5) and LP (9) stay below every cortical area. With the family 1 to 50, five cortical areas reach 46 to 49.

## fig7c-static-long-term-across-mice

> Long-term adaptation to static gratings (5- to 20-back) is consistent across mice.

In [ ]:
for pooled_i, lab in [(0, "trials pooled over 5 to 20"), (1, "rates averaged over 5 to 20")]:
    print(lab)
    consistency(mouse_rates(SG, A.CORTEX, range(5, 21), A.SG_WIN)[pooled_i], "  (a) cortex")
    consistency(mouse_rates(SG, ["V1"], range(5, 21), A.SG_WIN)[pooled_i], "  (b) V1")
    consistency(mouse_rates(SG, A.THALAMUS, range(5, 21), A.SG_WIN)[pooled_i], "  thalamus (reported only)")

**Result: partially reproduced.** For 5- to 20-back, 30 of 32 mice have the cortical orthogonal rate above the repeat rate (p = 2.5e-07) and 29 of 32 for V1 (p = 2.6e-06), so (b) holds and (a) does not under both readings of the group.

## fig8-no-spurious-long-term

> A simulated neuron with only 1-back adaptation, shown the experiments' own stimulus sequences,
> shows no spurious adaptation to stimuli further back, for drifting or static gratings.

For each session's sequence (blanks holding positions, pairs within blocks), a neuron that
responds equally to every grating, reduced by the empirical V1 1-back ratio of that stimulus set
when the 1-back grating had the same orientation (Figure 8 legend). Noise readings: (a) Poisson
counts at the V1 median rate on orthogonal 1-back trials, the claim's seed; (b) no noise. The
simulated responses are analysed as the data (pooled ratio per n). 95% CI across the 32
sequences (t interval of the mean log ratio). Ranges of n: (c) 2 to 10, (d) 2 to 50.

In [ ]:
def simulate(stim, truth, rate, win, rng, noise):
    out = []
    for s in S:
        table = s[stim]
        ori, blk, order = A.sequence(table, None, "hold", "within")
        rep1 = np.zeros(len(ori), bool)
        r_, o_, b_ = A.pair_masks(ori, blk, 1)
        rep1[1:] = r_
        lam = rate * win * np.where(rep1, truth, 1.0)
        c = rng.poisson(lam) if noise else lam
        counts = np.zeros(len(table)); counts[order] = c
        nb, _ = A.nback_rates(counts[None, :], table, 50, "hold", "within")
        out.append(np.log(A.ratios(nb, "pooled")[0]))
    return np.array(out)
s8 = seed("fig8-no-spurious-long-term")
def sim_eval(L, truth):
    L = np.where(np.isfinite(L), L, np.nan)
    m = np.nanmean(L, 0); se = np.nanstd(L, 0, ddof=1) / np.sqrt(np.sum(np.isfinite(L), 0))
    lo, hi = np.exp(m - stats.t.ppf(0.975, 31) * se), np.exp(m + stats.t.ppf(0.975, 31) * se)
    ok1 = lo[0] - 1e-9 <= truth <= hi[0] + 1e-9
    exc = [n for n in range(2, 51) if not (lo[n - 1] <= 1 <= hi[n - 1])]
    return dict(one_back=np.exp(m[0]), one_low=lo[0], one_high=hi[0], truth_in=ok1,
                excl_2_10=[n for n in exc if n <= 10], excl_2_50=exc,
                excl_2_10_ratios=[round(float(np.exp(m[n - 1])), 4) for n in exc if n <= 10],
                min_2_50=np.exp(m[1:]).min(), max_2_50=np.exp(m[1:]).max(),
                max_dev_2_10=float(np.max(np.abs(np.exp(m[1:10]) - 1))))
SIM = {}
for stim, ar, d, win in [("dg", ar_dg, DG, A.DG_WIN), ("sg", ar_sg, SG, A.SG_WIN)]:
    truth = A.region_value(ar[d["area"] == "V1", 0], "geo")
    v1 = d["area"] == "V1"
    rate = float(np.median(d["nb"]["os"][v1, 0].sum(1) / d["nb"]["on"][v1, 0].sum(1) / win))
    print(f"{'drifting' if stim == 'dg' else 'static'}: ground truth {truth:.4f}, rate {rate:.2f} Hz")
    for noise, k in [(True, 0), (True, 1000), (True, 2000), (True, 3000), (True, 4000), (False, 0)]:
        L = simulate(stim, truth, rate, win, np.random.default_rng(s8 + k + (0 if stim == "dg" else 1)), noise)
        SIM[(stim, noise, k)] = r_ = sim_eval(L, truth)
        print(f"  {'Poisson, seed + ' + str(k) if noise else 'no noise'}: " + ", ".join(
            f"{a}={round(b, 4) if isinstance(b, float) else b}" for a, b in r_.items()))
def fig8_reading(noise, k, rng_n):
    res = {}
    for stim in ["dg", "sg"]:
        r_ = SIM[(stim, noise, k)]
        ex = r_["excl_2_10"] if rng_n == "c" else r_["excl_2_50"]
        res[stim] = r_["truth_in"] and not ex
    return res["dg"] and res["sg"]
for noise in [True, False]:
    for rng_n in ["c", "d"]:
        ks = [0, 1000, 2000, 3000, 4000] if noise else [0]
        print(f"reading ({'a' if noise else 'b'}, {rng_n}): criterion met under seed +k: "
              + ", ".join(f"{k}: {fig8_reading(noise, k, rng_n)}" for k in ks))
for noise in [True, False]:
    part = all(len(SIM[(st, noise, 0)]["excl_2_10"]) <= 1 and all(abs(x - 1) <= 0.01 for x in SIM[(st, noise, 0)]["excl_2_10_ratios"]) for st in ["dg", "sg"])
    print(f"partial rule (no more than one n in 2-10 excluding 1, within 0.01 of 1), {'Poisson' if noise else 'no noise'}: {part}")

In [ ]:
# Seed dependence of reading (a): 200 further Poisson seeds, each simulated exactly as above, and
# the number of excluding n expected by chance (5% of the n per stimulus set if the CIs are exact).
def sim_weights(stim):
    """Per session, the 1-back repeat flags and, for n = 1 to 50, the positions of repeat and
    orthogonal pairs, so that the pooled ratio of `simulate` is a ratio of two weighted sums."""
    W = []
    for s in S:
        ori, blk, order = A.sequence(s[stim], None, "hold", "within")
        rep1 = np.zeros(len(ori), bool); rep1[1:] = A.pair_masks(ori, blk, 1)[0]
        wr = np.zeros((len(ori), 50)); wo = np.zeros((len(ori), 50))
        for n in range(1, 51):
            r_, o_, _ = A.pair_masks(ori, blk, n)
            wr[n:, n - 1] = r_ / r_.sum(); wo[n:, n - 1] = o_ / o_.sum()
        W.append((rep1, wr, wo))
    return W
def simulate_fast(W, truth, rate, win, rng):
    out = []
    for rep1, wr, wo in W:
        c = rng.poisson(rate * win * np.where(rep1, truth, 1.0)).astype(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            out.append(np.log((c @ wr) / (c @ wo)))
    return np.array(out)
P8 = {}
for stim, ar, d, win in [("dg", ar_dg, DG, A.DG_WIN), ("sg", ar_sg, SG, A.SG_WIN)]:
    truth = A.region_value(ar[d["area"] == "V1", 0], "geo")
    v1 = d["area"] == "V1"
    rate = float(np.median(d["nb"]["os"][v1, 0].sum(1) / d["nb"]["on"][v1, 0].sum(1) / win))
    W = sim_weights(stim)
    chk = sim_eval(simulate_fast(W, truth, rate, win, np.random.default_rng(s8 + (0 if stim == "dg" else 1))), truth)
    assert chk["excl_2_50"] == SIM[(stim, True, 0)]["excl_2_50"]   # the same as `simulate`
    P8[stim] = [sim_eval(simulate_fast(W, truth, rate, win, np.random.default_rng(s8 + 10_000 + 2 * j + (0 if stim == "dg" else 1))), truth)
                for j in range(200)]
def meets(r_, rng_n, partial=False):
    ex = r_["excl_2_10"] if rng_n == "c" else r_["excl_2_50"]
    if partial:
        return r_["truth_in"] and len(r_["excl_2_10"]) <= 1 and all(abs(x - 1) <= 0.01 for x in r_["excl_2_10_ratios"])
    return r_["truth_in"] and not ex
for rng_n in ["c", "d"]:
    both = np.mean([meets(a, rng_n) and meets(b, rng_n) for a, b in zip(P8["dg"], P8["sg"])])
    print(f"reading (a, {rng_n}): criterion met in {100 * both:.1f}% of 200 seeds "
          f"(drifting alone {100 * np.mean([meets(a, rng_n) for a in P8['dg']]):.1f}%, static alone {100 * np.mean([meets(b, rng_n) for b in P8['sg']]):.1f}%)")
part = np.mean([meets(a, "c", True) and meets(b, "c", True) for a, b in zip(P8["dg"], P8["sg"])])
print(f"partial rule met in {100 * part:.1f}% of 200 seeds")
for stim in ["dg", "sg"]:
    e10 = [len(r_["excl_2_10"]) for r_ in P8[stim]]; e50 = [len(r_["excl_2_50"]) for r_ in P8[stim]]
    dev = [abs(x - 1) for r_ in P8[stim] for x in r_["excl_2_10_ratios"]]
    print(f"{'drifting' if stim == 'dg' else 'static'}: excluding n per seed, 2 to 10 mean {np.mean(e10):.2f} (chance 0.45), "
          f"2 to 50 mean {np.mean(e50):.2f} (chance 2.45); truth inside the 1-back CI in {sum(r_['truth_in'] for r_ in P8[stim])} of 200; "
          f"excluding ratios within {max(dev) if dev else 0:.4f} of 1")
print("chance that no n of 9 (49) excludes 1 in both sets, if independent: "
      f"{0.95 ** 18:.3f} ({0.95 ** 98:.4f})")
for k in [1000, 2000, 3000, 4000]:
    print(f"partial rule under seed + {k}:", all(len(SIM[(st, True, k)]["excl_2_10"]) <= 1 and
          all(abs(x - 1) <= 0.01 for x in SIM[(st, True, k)]["excl_2_10_ratios"]) for st in ["dg", "sg"]))


**Result: not reproduced.** The simulated neuron's 1-back interval contains the ground truth in each of the six runs above; over the 200 further Poisson seeds it does in 192 (drifting) and 189 (static). Under the claim's seed with Poisson noise (a), no drifting n from 2 to 10 excludes 1, but static 5- and 6-back do (0.986, 0.987); with no noise (b), static passes and drifting 2- and 6-back exclude 1 (0.996, 1.003). No reading meets the criterion or its partial rule under the claim's seed; reading (a, c) meets it under one of the four other seeds. The spurious ratios are small: within 0.0037 of 1 for drifting and 0.0007 for static over n = 2 to 10 without noise, against V1 drifting ratios of 0.918 to 0.984 at 2- to 8-back. Over 200 further Poisson seeds, reading (a, c) meets the criterion in 38.0% and the partial rule in 41.5%, reading (a, d) in none; the n whose interval excludes 1 number 0.36 (drifting) and 0.47 (static) per seed over 2 to 10, and 2.57 and 2.46 over 2 to 50, close to the 0.45 and 2.45 expected by chance from 95% intervals, and if exclusions were independent no n of 9 would exclude 1 in both sets in 39.7% of seeds. The failures under reading (a) are thus those of a criterion that asks every interval to contain 1, which noise alone breaks about as often as the data show (a question to a reader, questions/fritsche-2022.yaml). Under the four other seeds the partial rule holds at +1000 and +4000. Without noise the exclusions (drifting 0.996 at 2-back and 1.003 at 6-back) come from the sequences themselves, not from noise.


## robust-threshold-preferred-window

> The results are qualitatively similar with a responsiveness threshold of 2.5 or 10 Hz, when
> only trials at the neuron's preferred orientation are used (drifting gratings), and with a 0
> to 100 ms window for static gratings.

Each variant: V1 adaptation significant beyond 1-back (FDR as in the main analysis: n = 1 to 10
for drifting, 1 to 20 for static, reading (a) of fig7a), V1 better fit by the double exponential
(F test on the mean, p < 0.05), and for drifting gratings no LGN n beyond 2 significant. The
preferred-orientation variant uses trials whose current orientation is the unit's preferred and
whose n-back is the preferred (repeat) or its orthogonal (Methods).

In [ ]:
def variant_eval(ar, area, stim, label):
    n_fam = 10 if stim == "dg" else 20
    _, sig = nback_tests(ar, area, "V1", range(1, n_fam + 1))
    lr = np.log(ar[area == "V1"]); lr = np.where(np.isfinite(lr), lr, np.nan)
    m = np.nanmean(lr, 0)
    dd, ss = A.fit_models(m, -1)
    F, p = A.ftest(ss[1], dd[1], len(m))
    row = dict(variant=label, V1_units=int((area == "V1").sum()),
               V1_nonfinite=int(np.sum(~np.isfinite(np.log(ar[area == "V1"])))),
               V1_sig=sig, V1_beyond_1=any(n > 1 for n in sig), V1_tau_slow=dd[0][3], F_p=p,
               double_better=p < 0.05)
    if stim == "dg":
        _, sl = nback_tests(ar, area, "LGN", range(1, 11))
        row.update(LGN_units=int((area == "LGN").sum()), LGN_sig=sl, LGN_ok=all(n <= 2 for n in sl))
    row["meets"] = row["V1_beyond_1"] and row["double_better"] and row.get("LGN_ok", True)
    return row
rows = []
for thr in [2.5, 10.0]:
    kw = dict(MAIN, thr=thr)
    d = build("dg", sel_kw=kw); rows.append(variant_eval(A.ratios(d["nb"], "pooled"), d["area"], "dg", f"drifting {thr} Hz"))
    d = build("sg", sel_kw=kw); rows.append(variant_eval(A.ratios(d["nb"], "pooled"), d["area"], "sg", f"static {thr} Hz"))
# Preferred orientation only (drifting).
pi = np.array([int(np.flatnonzero(DG["oris"] == p)[0]) for p in DG["pref"]])
u = np.arange(len(pi))
nbp = {k: v[u, :, pi][:, :, None] for k, v in DG["nb"].items()}
nbp = {k: v.reshape(len(pi), -1, 1) for k, v in nbp.items()}
with np.errstate(divide="ignore", invalid="ignore"):
    ar_pref = A.ratios(nbp, "pooled")
rows.append(variant_eval(ar_pref, DG["area"], "dg", "drifting, preferred orientation only"))
d = build("sg", short_counts=True)
rows.append(variant_eval(A.ratios(d["nb"], "pooled"), d["area"], "sg", "static 0-100 ms, units selected on 0-250 ms"))
d = build("sg", win=A.SG_WIN_SHORT)
rows.append(variant_eval(A.ratios(d["nb"], "pooled"), d["area"], "sg", "static 0-100 ms, units selected on 0-100 ms"))
rob = pd.DataFrame(rows)
print(rob.round(4).to_string())
print(f"variants meeting the reading: {int(rob.meets.sum())} of {len(rob)}")

**Result: reproduced.** All seven variants meet the reading: V1 adaptation is significant beyond 1-back, V1 favours the double exponential (F test p <= 0.0105), and for drifting gratings LGN is significant only at 1-back. This holds at 2.5 and 10 Hz for both stimulus sets, for the preferred-orientation trials, and for static 0 to 100 ms rates with units selected on 0 to 250 ms (545 V1 units) or on 0 to 100 ms (97 V1 units).

## Summary

| Claim | Outcome |
|---|---|
| units-responsive-count | reproduced |
| fig1d-v1-1back | reproduced |
| fig1d-extrastriate-1back | reproduced |
| fig1d-thalamus-1back | reproduced |
| fig1e-consistent-across-mice | partially reproduced |
| fig2-v1-tuning-main-effect | reproduced |
| fig2-v1-tuning-interaction | reproduced |
| fig2-v1-adaptation-at-all-orientations | reproduced |
| fig2-thalamus-no-tuning-dependence | reproduced |
| fig3a-v1-long-term | reproduced |
| fig3a-v1-double-exponential | reproduced |
| fig3a-v1-half-life | reproduced |
| fig4-cumulative-adaptation | reproduced |
| fig4b-linear-accumulation | reproduced |
| fig3b-thalamus-short-term | reproduced (rests on an assumption) |
| v1-vs-thalamus-welch | reproduced |
| fig3c-mice-long-term | reproduced |
| decay-cortex-double-better | reproduced |
| table1-extrastriate-tau-slow | reproduced |
| decay-thalamus-single | reproduced (rests on an assumption) |
| fig5-repeat-suppression | not reproduced |
| fig6c-static-v1-1back | reproduced |
| fig6c-static-other-areas-1back | reproduced |
| fig6d-static-consistent-across-mice | partially reproduced |
| fig7a-static-v1-long-term | reproduced |
| fig7b-static-cortex-decay | reproduced |
| fig7-static-lgn-single | reproduced |
| fig7b-static-areas-long-term | reproduced |
| fig7c-static-long-term-across-mice | partially reproduced |
| fig8-no-spurious-long-term | not reproduced (seed sensitive) |
| robust-threshold-preferred-window | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's code is restricted and was not read.
The open details of the claims file were each run; the adopted main reading (pooled per-unit
ratio, geometric region value, blank trials holding positions, pairs within blocks, gray mean
+ 1 SD of the blank trials, preferred orientation pooled, `noise` units left out) is the one the
unit counts single out, and it gives the paper's 1-back values to within 0.01 for drifting gratings and for static
gratings in cortex; for static gratings in the thalamus it gives LGN 0.957 against 0.93 and LP
0.900 against 0.85. Two outcomes marked above rest on assumptions. Leaving out the `noise`
units is one: with them kept, LGN gains a significant 7-back and its single-exponential tau
falls to 0.000, outside the interval of decay-thalamus-single, and no reading of
fig3b-thalamus-short-term meets its criterion (RL stops at 5-back). fig3b also rests on a
second assumption, that significant adaptation counts only ratios below 1; read two-sided, it
would be partially reproduced. The invalid
intervals listed in the files are not applied; leaving out the presentations inside them leaves
the unit counts and 1-back ratios within tolerance, keeps fig3b's across-blocks reading passing,
and adds a significant LGN 7-back under the main reading. The simulated neuron of Figure 8 needs a rate and
a noise model the paper does not give; the V1 median rate with Poisson counts, and no noise, were
run. Its criterion asks every interval to contain 1, which Poisson noise breaks at about the
rate expected by chance (38% of 200 seeds pass); this is asked of a reader.

What was found about the dataset. The files hold the 32 mice and both stimulus sets the paper
describes. The units table holds every sorted unit, with a `quality` column marking 20182 of
74658 as `noise` and quality metric columns, none of them described in the file; the paper's
counts are matched when the `noise` units are left out, as the AllenSDK does. Three sessions
hold 598 drifting gratings in place of 600. Thirteen sessions list probe-tagged invalid
intervals, some inside the grating epochs. Units on channels labelled `grey` or `VIS` have no
area.